# Tuần 2 (tiếp tục): Judge trên dev, phiếu gán nhãn 100 câu, và nhúng nốt C5 / C5_nohead có checkpoint

Notebook sinh tự động bởi `scripts/build_colab_notebook.py` từ `src/ragvi/` (đã nâng cấp `embed()` hỗ trợ checkpoint mỗi 20.000 chunk và tự động bật `bf16`/`fp16` + `batch_size` theo GPU T4/L4/A100).
Các kết quả Ngày 8 (`GEN_MODEL = Qwen/Qwen2.5-7B-Instruct`, `JUDGE_MODEL = Qwen/Qwen3-8B`), Ngày 9, Reranker R1 và 6/8 lượt nhúng `BAAI/bge-m3` đã lưu trên Drive từ notebook 04.

### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Cài thư viện

Cài các thư viện cần thiết cho retrieval, rerank, generate và judge 4-bit.

In [ ]:
%pip install -q rank_bm25 underthesea sentence-transformers datasets bitsandbytes accelerate

### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Kết nối Google Drive

Kết nối Google Drive chứa dữ liệu và các file đệm từ notebook 02–04.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

### [TÁI SỬ DỤNG] Đường dẫn và kiểm tra GPU

In thông tin GPU đang kết nối (T4 / L4 / A100) và khởi tạo đường dẫn.

In [ ]:
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Runtime không có GPU. Đổi sang runtime GPU (T4/L4/A100).')
prop = torch.cuda.get_device_properties(0)
print(f'GPU: {prop.name} | VRAM: {prop.total_memory / (1024**3):.1f} GB | bf16: {torch.cuda.is_bf16_supported()}')

PROJECT_DIR = Path('/content/drive/MyDrive/Vietnamese-RAG-Research')
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Không thấy {PROJECT_DIR}. Hãy chạy cell kết nối Drive trước.')
P = PROJECT_DIR / 'data' / 'processed'
Z = P / 'zalo_legal'
EVAL = PROJECT_DIR / 'data' / 'eval'
O = PROJECT_DIR / 'results' / 'tuan1'
R = O / 'retrieval'
O2 = PROJECT_DIR / 'results' / 'tuan2'
GEN = O2 / 'generate'
JUDGE = O2 / 'judge'
SHEETS2 = O2 / 'review'
GEN_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
JUDGE_MODEL = 'Qwen/Qwen3-8B'
for d in (R, GEN, JUDGE, SHEETS2, Path('/content/src/ragvi')):
    d.mkdir(parents=True, exist_ok=True)
print(P, O, O2, EVAL, sep='\n')

### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/__init__.py`

Chép mã nguồn mới nhất (có checkpoint/resume embedding + fp16/bf16) vào `/content/src/ragvi/`.

In [ ]:
%%writefile /content/src/ragvi/__init__.py


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/chunking.py`

Chép mã nguồn mới nhất (có checkpoint/resume embedding + fp16/bf16) vào `/content/src/ragvi/`.

In [ ]:
%%writefile /content/src/ragvi/chunking.py
"""Phân đoạn: C1 (cố định), C3 (theo câu), C4 (nguyên tài liệu), C5 ViStructChunk (Điều → Khoản → Điểm).

Mọi hàm trả về danh sách (chunk_id, doc_id, start, end, text); start/end là vị trí ký tự trong
tài liệu gốc và text là lát cắt nguyên văn text[start:end] (riêng C5 có thể gắn thêm tiêu đề cha
ở phía trước). Token được đếm bằng một tokenizer tham chiếu truyền từ ngoài vào (D7: BAAI/bge-m3).

Chạy (Zalo):   PYTHONPATH=src python -m ragvi.chunking <processed>/zalo_legal --methods c1,c3,c4,c5,c5_nohead
Chạy (ViQuAD): PYTHONPATH=src python -m ragvi.chunking <eval>/viquad/corpus.jsonl   (chỉ c1, c3, c4)
Phiếu đối chiếu tay C5: PYTHONPATH=src python -m ragvi.chunking <processed>/zalo_legal --review c5_review.md
"""
import argparse
import json
import math
import random
import re
import statistics
from bisect import bisect_left
from pathlib import Path

SIZE = 256
EMBED_LIMIT = 512  # mô hình nhúng cắt ở 512 token (D7)
METHODS = ('c1', 'c3', 'c4', 'c5', 'c5_nohead')
CLAUSE = re.compile(r'^(\d+)\.\s', re.M)      # Khoản: "1. ..."
POINT = re.compile(r'^([a-zđ])\)\s', re.M)    # Điểm: "a) ..."


def _offsets(text, tok):
    """Trả về (offset_mapping, danh sách vị trí bắt đầu của từng token)."""
    off = tok(text, add_special_tokens=False, return_offsets_mapping=True)['offset_mapping']
    return off, [o[0] for o in off]


def _ntok(starts, s, e):
    """Số token bắt đầu trong khoảng ký tự [s, e)."""
    return bisect_left(starts, e) - bisect_left(starts, s)


def _fixed_spans(off, starts, s, e, size):
    """Cắt khoảng [s, e) thành các đoạn `size` token, không chồng lấn.

    XLM-R (bge-m3) có token "▁" đứng riêng mang offset trùng ký tự đầu của token sau (ví dụ "ỦY", "QĐ", "≥"),
    nên đầu cuối của mỗi đoạn bị chặn tại điểm bắt đầu của đoạn kế tiếp.
    """
    i0, i1 = bisect_left(starts, s), bisect_left(starts, e)
    return [(max(off[i][0], s), min(off[min(i + size, i1) - 1][1], off[i + size][0] if i + size < i1 else e, e))
            for i in range(i0, i1, size)]


def _sentence_spans(text, s, e, sent_split):
    """Vị trí (start, end) của từng câu trong text[s:e]; phần chữ hàm tách câu bỏ sót được giữ lại."""
    spans, pos = [], s
    for sent in sent_split(text[s:e]):
        sent = sent.strip()
        if not sent:
            continue
        a = text.find(sent, pos, e)
        b = a + len(sent)
        if a < 0:  # hàm tách câu đã đổi khoảng trắng: so khớp bỏ qua khoảng trắng
            m = re.compile(r'\s*'.join(map(re.escape, sent.split()))).search(text, pos, e)
            if not m:
                raise ValueError(f'Không tìm thấy câu trong văn bản gốc: {sent[:80]!r}')
            a, b = m.span()
        a -= len(text[pos:a].lstrip())  # chữ nằm giữa hai câu thuộc về câu sau
        spans.append((a, b))
        pos = b
    tail = text[pos:e]
    if tail.strip():
        spans.append((e - len(tail.lstrip()), pos + len(tail.rstrip())))
    return spans


def _sentence_merge_spans(text, off, starts, s, e, size, sent_split):
    """C3 trên khoảng [s, e): gộp câu tới `size` token; câu dài hơn `size` thì cắt bằng C1."""
    out, cur = [], None
    for a, b in _sentence_spans(text, s, e, sent_split):
        if cur and _ntok(starts, cur[0], b) <= size:
            cur = (cur[0], b)
            continue
        if cur:
            out.append(cur)
            cur = None
        if _ntok(starts, a, b) > size:
            out += _fixed_spans(off, starts, a, b, size)
        else:
            cur = (a, b)
    return out + [cur] if cur else out


def _default_sent_split():
    from underthesea import sent_tokenize
    return sent_tokenize


def _records(doc_id, text, spans):
    return [(f'{doc_id}#{i}', doc_id, s, e, text[s:e]) for i, (s, e) in enumerate(spans)]


def chunk_fixed(doc_id, text, tok, size=SIZE):
    """C1: mỗi chunk `size` token, không chồng lấn, cắt theo vị trí ký tự do tokenizer trả về."""
    off, starts = _offsets(text, tok)
    return _records(doc_id, text, _fixed_spans(off, starts, 0, len(text), size))


def chunk_sentence(doc_id, text, tok, size=SIZE, sent_split=None):
    """C3: tách câu (mặc định `underthesea.sent_tokenize`), gộp câu tới `size` token."""
    off, starts = _offsets(text, tok)
    spans = _sentence_merge_spans(text, off, starts, 0, len(text), size, sent_split or _default_sent_split())
    return _records(doc_id, text, spans)


def chunk_document(doc_id, text):
    """C4: nguyên tài liệu (đoạn ViQuAD / nguyên điều luật Zalo) là một chunk."""
    return [(f'{doc_id}#0', doc_id, 0, len(text), text)]


def has_clause(body):
    """Điều luật có ít nhất một khoản được nhận diện hay không."""
    return bool(CLAUSE.search(body))


def struct_heading(doc_id, title):
    """Hai phần tiêu đề cha của C5: ("Văn bản X", "Văn bản X, <tiêu đề Điều>")."""
    doc = 'Văn bản ' + doc_id.rsplit('+', 1)[0].upper()
    return doc, ', '.join(filter(None, [doc, title.strip()]))


def chunk_struct(doc_id, title, body, tok, size=SIZE, heading=True, sent_split=None):
    """C5 ViStructChunk cho một điều luật; `heading=False` là biến thể không tiêu đề (cùng ranh giới).

    Tài liệu gốc là `title + "\\n" + body` (giống `ragvi.data.load_zalo_corpus`), start/end tính trên
    chuỗi đó. Đơn vị chunk là Khoản; Khoản dài hơn `size` token thì tách theo Điểm; phần mở đầu trước
    khoản 1 (kèm dòng tiêu đề Điều) là một chunk riêng; điều luật không có khoản là một chunk, trùng C4.
    Mảnh nào vẫn dài hơn `size` token thì cắt bằng C3.
    Tiêu đề cha: "Văn bản 01/2009/TT-BNN, Điều 1. ..., Khoản 2[, Điểm a]: " + nội dung. Chunk bắt đầu
    từ đầu tài liệu đã chứa dòng tiêu đề Điều nên chỉ gắn "Văn bản ..., ".
    """
    text = title + '\n' + body
    off, starts = _offsets(text, tok)
    doc, full = struct_heading(doc_id, title)

    def cut(s, e, head):
        e = s + len(text[s:e].rstrip())
        if s >= e:
            return []
        if _ntok(starts, s, e) <= size:
            spans = [(s, e)]
        else:
            spans = _sentence_merge_spans(text, off, starts, s, e, size, sent_split or _default_sent_split())
        return [(a, b, f'{doc}, ' if a == 0 else f'{head}: ') for a, b in spans]

    # ponytail: nhận diện khoản chỉ bằng regex đầu dòng, không kiểm tra số khoản tăng dần;
    # nếu đối chiếu tay 30 điều (Ngày 7) thấy nhận nhầm danh sách đánh số thì thêm kiểm tra đó.
    clauses = list(CLAUSE.finditer(text, len(title) + 1))
    if not clauses:
        pieces = [(0, len(text), f'{doc}, ')] if len(off) <= size else cut(0, len(text), full)
    else:
        # Luôn giữ phần trước khoản 1, kể cả khi chỉ còn dòng tiêu đề Điều; bỏ đi thì c5_nohead mất tiêu đề.
        pieces = cut(0, clauses[0].start(), full)
        for m, nxt in zip(clauses, clauses[1:] + [None]):
            s, e = m.start(), nxt.start() if nxt else len(text)
            head = f'{full}, Khoản {m[1]}'
            points = list(POINT.finditer(text, s, e)) if _ntok(starts, s, e) > size else []
            if not points:
                pieces += cut(s, e, head)
                continue
            pieces += cut(s, points[0].start(), head)
            for p, q in zip(points, points[1:] + [None]):
                pieces += cut(p.start(), q.start() if q else e, f'{head}, Điểm {p[1]}')
    return [(f'{doc_id}#{i}', doc_id, s, e, (prefix if heading else '') + text[s:e])
            for i, (s, e, prefix) in enumerate(pieces)]


def chunk_stats(chunks, tok):
    """Số chunk và phân phối độ dài theo token (tính trên `text` của chunk, kể cả tiêu đề cha)."""
    # ponytail: token hóa lại từng chunk; nếu chậm trên toàn kho thì token hóa theo lô.
    # `chunks` có thể là generator: chỉ giữ danh sách độ dài, không giữ chunk trong bộ nhớ.
    lengths = sorted(len(_offsets(c[4], tok)[0]) for c in chunks)
    if not lengths:
        return {'n_chunks': 0, 'tok_mean': 0, 'tok_median': 0, 'tok_p95': 0, 'tok_max': 0, 'over_512': 0.0}
    return {'n_chunks': len(lengths), 'tok_mean': statistics.fmean(lengths),
            'tok_median': statistics.median(lengths),
            'tok_p95': lengths[math.ceil(0.95 * len(lengths)) - 1], 'tok_max': lengths[-1],
            'over_512': sum(n > EMBED_LIMIT for n in lengths) / len(lengths)}


def clause_rate(bodies):
    """Tỷ lệ điều luật có ít nhất một khoản được nhận diện."""
    bodies = list(bodies)
    return sum(map(has_clause, bodies)) / len(bodies) if bodies else 0.0


REVIEW_GUIDE = """# Phiếu đối chiếu tay C5 (ViStructChunk)

Hướng dẫn: với mỗi điều luật, đọc nguyên văn rồi so với danh sách chunk C5 bên dưới.
Đánh "đúng" nếu mỗi khoản (và điểm, khi khoản dài) thành đúng một chunk với tiêu đề cha đúng số khoản/điểm.
Đánh "khoản nhận nhầm" nếu một dòng không phải khoản (ví dụ danh sách đánh số trong một khoản) bị tách thành khoản; "bỏ sót khoản/điểm" nếu một khoản hay điểm có trong nguyên văn mà không thành chunk riêng.
Lỗi khác (tiêu đề cha sai, cắt giữa câu bất thường...) ghi vào ô "khác".
"""


def review_sheet(docs, titles, n=30, seed=0, tok=None, size=SIZE, sent_split=None, max_chars=3000):
    """Markdown gồm `n` điều luật chọn ngẫu nhiên theo `seed`: nguyên văn và danh sách chunk C5."""
    out = [REVIEW_GUIDE, f'Seed: {seed}; số điều luật: {min(n, len(docs))}.\n']
    for k, doc_id in enumerate(random.Random(seed).sample(sorted(docs), min(n, len(docs))), 1):
        text = docs[doc_id]
        shown = text if len(text) <= max_chars else (
            text[:max_chars] + f'\n[... ĐÃ CẮT: chỉ hiện {max_chars}/{len(text)} ký tự đầu]')
        out += [f'## {k}. `{doc_id}`\n', '````text', shown, '````\n', 'Chunk C5:\n']
        for cid, _, s, e, ctext in chunk_struct(doc_id, titles[doc_id], text[len(titles[doc_id]) + 1:], tok,
                                                size, sent_split=sent_split):
            head = ctext[:len(ctext) - (e - s)]
            out.append(f'- `{cid}` [{s}:{e}] **{head.strip()}** {text[s:e][:120]!r}')
        out.append('\nĐánh giá: [ ] đúng  [ ] khoản nhận nhầm  [ ] bỏ sót khoản/điểm  [ ] khác:\n')
    return '\n'.join(out)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument('processed_dir', help='thư mục <processed>/zalo_legal, hoặc file corpus.jsonl (ViQuAD)')
    parser.add_argument('--methods', help=f'mặc định {",".join(METHODS)} (Zalo) hoặc c1,c3,c4 (ViQuAD)')
    parser.add_argument('--tokenizer', default='BAAI/bge-m3')
    parser.add_argument('--size', type=int, default=SIZE)
    parser.add_argument('--review', help='ghi phiếu đối chiếu tay C5 ra file Markdown này thay vì in thống kê')
    parser.add_argument('--review-n', type=int, default=30)
    parser.add_argument('--seed', type=int, default=0)
    args = parser.parse_args()
    viquad = args.processed_dir.endswith('.jsonl')
    methods = (args.methods or ('c1,c3,c4' if viquad else ','.join(METHODS))).split(',')
    allowed = ('c1', 'c3', 'c4') if viquad else METHODS
    if set(methods) - set(allowed):
        parser.error(f'Cách phân đoạn không hợp lệ: {sorted(set(methods) - set(allowed))}; chọn trong {allowed}')
    if viquad and args.review:
        parser.error('Phiếu đối chiếu C5 chỉ dùng cho kho Zalo')

    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained(args.tokenizer)
    if viquad:
        with open(args.processed_dir, encoding='utf-8') as f:
            docs = {r['doc_id']: r['text'] for r in map(json.loads, f)}
    else:
        from datasets import load_from_disk

        from ragvi.data import load_zalo_corpus

        docs = load_zalo_corpus(args.processed_dir)
        # C5 cần tiêu đề riêng, dict {doc_id: title + "\n" + text} không tách được chắc chắn nên đọc thẳng cột title.
        corpus = load_from_disk(str(Path(args.processed_dir) / 'corpus')).select_columns(['id', 'title'])
        titles = {row['id']: row['title'] for row in corpus}
        bodies = {}
        for doc_id, text in docs.items():
            if not text.startswith(titles[doc_id] + '\n'):
                raise ValueError(f'Tài liệu {doc_id} không có dạng title + "\\n" + text')
            bodies[doc_id] = text[len(titles[doc_id]) + 1:]

    if args.review:
        Path(args.review).write_text(review_sheet(docs, titles, args.review_n, args.seed, tok, args.size),
                                     encoding='utf-8')
        print(f'Đã ghi phiếu đối chiếu {min(args.review_n, len(docs))} điều luật (seed {args.seed}) vào {args.review}')
        raise SystemExit

    run = {
        'c1': lambda d: chunk_fixed(d, docs[d], tok, args.size),
        'c3': lambda d: chunk_sentence(d, docs[d], tok, args.size),
        'c4': lambda d: chunk_document(d, docs[d]),
        'c5': lambda d: chunk_struct(d, titles[d], bodies[d], tok, args.size),
        'c5_nohead': lambda d: chunk_struct(d, titles[d], bodies[d], tok, args.size, heading=False),
    }
    print(f'Số tài liệu: {len(docs)}')
    print(f'{"cách":<10}{"số chunk":>10}{"tb":>9}{"trung vị":>10}{"p95":>8}{"max":>8}{">512":>8}')
    for name in methods:
        st = chunk_stats((c for d in docs for c in run[name](d)), tok)
        print(f'{name:<10}{st["n_chunks"]:>10}{st["tok_mean"]:>9.1f}{st["tok_median"]:>10.1f}'
              f'{st["tok_p95"]:>8}{st["tok_max"]:>8}{st["over_512"]:>8.2%}')
    if not viquad:
        print(f'Tỷ lệ điều luật có ít nhất một khoản: {clause_rate(bodies.values()):.1%}')
    if 'c5' in methods:
        # Chẩn đoán: độ dài token của phần tiêu đề cha chung cho mọi chunk của một điều ("Văn bản X, <title>").
        heads = sorted(((len(_offsets(struct_heading(d, titles[d])[1], tok)[0]), d) for d in docs), reverse=True)
        n = sorted(h for h, _ in heads)
        print(f'Tiêu đề cha C5 (token, mỗi điều luật): trung vị {statistics.median(n):.1f}, '
              f'p95 {n[math.ceil(0.95 * len(n)) - 1]}, max {n[-1]}')
        print('5 điều luật có tiêu đề cha dài nhất (token, doc_id, 200 ký tự đầu title):')
        for h, d in heads[:5]:
            print(f'  {h:>6}  {d}  {titles[d][:200]!r}')


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/data.py`

Chép mã nguồn mới nhất (có checkpoint/resume embedding + fp16/bf16) vào `/content/src/ragvi/`.

In [ ]:
%%writefile /content/src/ragvi/data.py
"""Dựng tập đánh giá Zalo Legal và ViQuAD theo quyết định D1 (reports/quyet-dinh-thiet-ke.md).

Chạy trên Colab, từ thư mục gốc của kho mã:
    PYTHONPATH=src python -m ragvi.data zalo <processed>/zalo_legal <thư mục đầu ra> --seed 0
    PYTHONPATH=src python -m ragvi.data viquad <processed>/viquad <thư mục đầu ra> --seed 0
"""
import argparse
import hashlib
import json
import random
import re
from pathlib import Path

# Thứ tự là thứ tự ưu tiên khi xếp nhóm vào từng phần.
SIZES = {'unanswerable': 120, 'dev': 100, 'answerable': 280}


def load_zalo_queries(processed_dir):
    """Trả về (questions: {qid: text}, gold: {qid: set(doc_id)}) cho các truy vấn có nhãn."""
    from datasets import load_from_disk

    root = Path(processed_dir)
    gold = {}
    for row in load_from_disk(str(root / 'qrels')):
        if row['score'] > 0:
            gold.setdefault(row['query-id'], set()).add(row['corpus-id'])

    questions = {}
    for row in load_from_disk(str(root / 'queries')):
        qid, text = row['id'], row['text']
        if qid not in gold:
            continue
        if questions.setdefault(qid, text) != text:
            raise ValueError(f'Truy vấn trùng mã nhưng khác nội dung: {qid}')
    return questions, gold


def zalo_doc_variants(doc_id):
    """Các biến thể mã tài liệu Zalo do lỗi chính tả nd-cp <-> nđ-cp trong kho gốc (ví dụ 24/2020/nd-cp+14)."""
    return {doc_id, doc_id.replace('/nđ-', '/nd-'), doc_id.replace('/nd-', '/nđ-')}


def expand_zalo_doc_ids(doc_ids, corpus=None):
    """Mở rộng tập doc_id với biến thể nd-cp <-> nđ-cp; nếu có `corpus` thì chỉ thêm biến thể có thật trong kho."""
    base = set(doc_ids)
    extra = {v for d in base for v in zalo_doc_variants(d)} - base
    if corpus is not None:
        extra &= set(corpus)
    return base | extra


def load_zalo_corpus(processed_dir, exclude=()):
    """Trả về {doc_id: title + "\\n" + text}; điều luật có text rỗng vẫn giữ (chỉ còn tiêu đề).

    exclude: các doc_id bị loại, ví dụ nội dung removed_doc_ids.json, để dựng kho thu hẹp (tự động loại cả
    bản trùng mã nd-cp <-> nđ-cp theo quyết định D6 ngày 18/11).
    """
    from datasets import load_from_disk

    exclude = expand_zalo_doc_ids(exclude)
    return {row['id']: row['title'] + '\n' + row['text']
            for row in load_from_disk(str(Path(processed_dir) / 'corpus')) if row['id'] not in exclude}


def gold_groups(gold):
    """Nhóm các truy vấn dùng chung điều luật vàng, trực tiếp hoặc bắc cầu (union-find)."""
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for qid, docs in gold.items():
        for doc in docs:
            parent[find(('q', qid))] = find(('d', doc.replace('/nd-', '/nđ-')))

    groups = {}
    for qid in gold:
        groups.setdefault(find(('q', qid)), []).append(qid)
    return sorted((sorted(g) for g in groups.values()), key=lambda g: g[0])


def split_zalo(gold, sizes=SIZES, seed=0):
    """Xếp trọn từng nhóm vào phần đầu tiên còn đủ chỗ; nhóm không vừa phần nào thì để dự trữ.

    Trả về (parts: {tên phần: [qid]}, group_of: {qid: chỉ số nhóm}).
    """
    groups = gold_groups(gold)
    random.Random(seed).shuffle(groups)
    parts = {name: [] for name in (*sizes, 'reserve')}
    group_of = {}
    for index, group in enumerate(groups):
        group_of.update((qid, index) for qid in group)
        name = next((n for n in sizes if len(parts[n]) + len(group) <= sizes[n]), 'reserve')
        parts[name] += group
    return parts, group_of


def build_zalo_eval(questions, gold, out_dir, sizes=SIZES, seed=0, tolerance=5):
    """Ghi test.jsonl, dev.jsonl, reserve.jsonl, removed_doc_ids.json, manifest.json.

    Từ chối ghi đè: tập đánh giá đã đóng băng thì không dựng lại vào cùng thư mục.
    """
    out = Path(out_dir)
    if out.exists() and any(out.iterdir()):
        raise FileExistsError(f'{out} đã có dữ liệu. Tập đánh giá đã đóng băng; hãy chọn thư mục khác.')

    parts, group_of = split_zalo(gold, sizes, seed)
    for name, target in sizes.items():
        if target - len(parts[name]) > tolerance:
            raise ValueError(f'Phần {name} chỉ có {len(parts[name])}/{target} câu; thử seed khác.')

    removed = sorted(set().union(*(gold[q] for q in parts['unanswerable'])))
    kept_gold = set().union(*(gold[q] for name in ('dev', 'answerable', 'reserve') for q in parts[name]))
    assert not kept_gold & set(removed), 'Điều luật bị loại vẫn là điều luật vàng của câu ở phần khác'

    def rows(names):
        return [{'qid': q, 'question': questions[q], 'answerable': name != 'unanswerable',
                 'gold_doc_ids': sorted(gold[q]), 'group': group_of[q]}
                for name in names for q in parts[name]]

    out.mkdir(parents=True, exist_ok=True)
    files = {'test.jsonl': rows(['answerable', 'unanswerable']), 'dev.jsonl': rows(['dev']),
             'reserve.jsonl': rows(['reserve'])}
    for name, data in files.items():
        (out / name).write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in data), encoding='utf-8')
    (out / 'removed_doc_ids.json').write_text(json.dumps(removed, ensure_ascii=False, indent=0), encoding='utf-8')

    manifest = {
        'dataset': 'zalo_legal', 'seed': seed, 'target_sizes': sizes,
        'actual_sizes': {name: len(qids) for name, qids in parts.items()},
        'labeled_queries': len(gold), 'groups': len(set(group_of.values())),
        'removed_docs': len(removed),
        'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(out.iterdir())},
    }
    (out / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest


def load_viquad(processed_dir):
    """Trả về danh sách dòng của cả ba split, mỗi dòng thêm khóa 'split'."""
    from datasets import load_from_disk

    return [{**row, 'split': split} for split, ds in load_from_disk(str(processed_dir)).items() for row in ds]


def passage_id(context):
    """Mã tài liệu của một đoạn ViQuAD: 16 ký tự hex đầu của SHA-256 nội dung đoạn."""
    return hashlib.sha256(context.encode('utf-8')).hexdigest()[:16]


def viquad_corpus(rows):
    """Trả về {doc_id: {'title', 'text'}} gồm các context duy nhất của mọi split (kể cả test)."""
    corpus = {}
    for row in rows:
        corpus.setdefault(passage_id(row['context']), {'title': row['title'], 'text': row['context']})
    return corpus


def locate_answers(context, answers):
    """Trả về [{'text', 'start', 'end'}] sao cho context[start:end] == text.

    answer_start lệch (ví dụ do bước làm sạch gộp khoảng trắng) thì lấy lần xuất hiện gần nhất;
    đáp án không còn nằm trong đoạn thì bỏ.
    """
    if not answers:
        return []
    pairs = (zip(answers['text'], answers['answer_start']) if isinstance(answers, dict)
             else ((a['text'], a['answer_start']) for a in answers))
    spans = set()
    for text, start in pairs:
        if not text:
            continue
        if context[start:start + len(text)] != text:
            found = [m.start() for m in re.finditer(re.escape(text), context)]
            if not found:
                continue
            start = min(found, key=lambda pos: abs(pos - start))
        spans.add((start, start + len(text)))
    return [{'text': context[s:e], 'start': s, 'end': e} for s, e in sorted(spans)]


def viquad_questions(rows):
    """Câu hỏi dùng được từ train và validation; qid kèm tên split vì id lặp lại giữa các split.

    Câu trả lời được mà không định vị được đáp án nào trong đoạn thì bị bỏ.
    """
    out = []
    for row in rows:
        if row['split'] not in ('train', 'validation'):
            continue
        answers = [] if row['is_impossible'] else locate_answers(row['context'], row['answers'])
        if not row['is_impossible'] and not answers:
            continue
        out.append({'qid': f"{row['split']}:{row['id']}", 'question': row['question'],
                    'answerable': not row['is_impossible'], 'gold_doc_id': passage_id(row['context']),
                    'title': row['title'], 'answers': answers})
    return sorted(out, key=lambda q: q['qid'])


def split_viquad(questions, sizes=SIZES, seed=0):
    """Tách theo title: dev lấy trọn các title ngẫu nhiên tới khi đủ câu; test lấy từ các title còn lại.

    Mỗi đoạn tối đa một câu (cả dev lẫn test). Trả về {tên phần: [câu hỏi]}.
    """
    rng = random.Random(seed)
    questions = list(questions)
    rng.shuffle(questions)
    titles = sorted({q['title'] for q in questions})
    rng.shuffle(titles)

    parts = {name: [] for name in sizes}
    dev_titles, used = set(), set()
    for title in titles:
        if len(parts['dev']) >= sizes['dev']:
            break
        dev_titles.add(title)
        for q in questions:
            if q['title'] == title and q['gold_doc_id'] not in used and len(parts['dev']) < sizes['dev']:
                used.add(q['gold_doc_id'])
                parts['dev'].append(q)

    # Đoạn nào thuộc title của dev thì không vào test, kể cả khi nó lặp lại dưới title khác.
    used |= {q['gold_doc_id'] for q in questions if q['title'] in dev_titles}
    for q in questions:
        name = 'answerable' if q['answerable'] else 'unanswerable'
        if q['title'] not in dev_titles and q['gold_doc_id'] not in used and len(parts[name]) < sizes[name]:
            used.add(q['gold_doc_id'])
            parts[name].append(q)
    return parts


def build_viquad_eval(rows, out_dir, sizes=SIZES, seed=0):
    """Ghi test.jsonl, dev.jsonl, corpus.jsonl, manifest.json; từ chối ghi đè như build_zalo_eval."""
    out = Path(out_dir)
    if out.exists() and any(out.iterdir()):
        raise FileExistsError(f'{out} đã có dữ liệu. Tập đánh giá đã đóng băng; hãy chọn thư mục khác.')

    questions = viquad_questions(rows)
    parts = split_viquad(questions, sizes, seed)
    for name, target in sizes.items():
        if len(parts[name]) != target:
            raise ValueError(f'Phần {name} chỉ có {len(parts[name])}/{target} câu; dữ liệu không đủ.')

    corpus = viquad_corpus(rows)
    out.mkdir(parents=True, exist_ok=True)
    files = {'test.jsonl': parts['answerable'] + parts['unanswerable'], 'dev.jsonl': parts['dev'],
             'corpus.jsonl': [{'doc_id': d, **corpus[d]} for d in sorted(corpus)]}
    for name, data in files.items():
        (out / name).write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in data), encoding='utf-8')

    manifest = {
        'dataset': 'viquad', 'seed': seed, 'target_sizes': sizes,
        'actual_sizes': {name: len(qs) for name, qs in parts.items()},
        'dev_answerable': sum(q['answerable'] for q in parts['dev']),
        'titles': {name: len({q['title'] for q in qs}) for name, qs in parts.items()},
        'usable_questions': len(questions),
        'labeled_rows': sum(row['split'] in ('train', 'validation') for row in rows),
        'corpus_passages': len(corpus),
        'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(out.iterdir())},
    }
    (out / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument('dataset', choices=['zalo', 'viquad'])
    parser.add_argument('processed_dir')
    parser.add_argument('out_dir')
    parser.add_argument('--seed', type=int, default=0)
    args = parser.parse_args()
    if args.dataset == 'zalo':
        result = build_zalo_eval(*load_zalo_queries(args.processed_dir), args.out_dir, seed=args.seed)
    else:
        result = build_viquad_eval(load_viquad(args.processed_dir), args.out_dir, seed=args.seed)
    print(json.dumps(result, ensure_ascii=False, indent=2))


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/generate.py`

Chép mã nguồn mới nhất (có checkpoint/resume embedding + fp16/bf16) vào `/content/src/ragvi/`.

In [ ]:
%%writefile /content/src/ragvi/generate.py
"""Sinh câu trả lời đầu-cuối cho cấu hình RAG, closed-book và oracle (Tuần 2: Ngày 8–9).

Chạy từ thư mục gốc của kho mã:
    PYTHONPATH=src python -m ragvi.generate <questions.jsonl> <units_path> <out.jsonl> --config oracle --model Qwen/Qwen3-8B
    PYTHONPATH=src python -m ragvi.generate <questions.jsonl> <units_path> <out.jsonl> --config closed_book --model Qwen/Qwen3-8B
    PYTHONPATH=src python -m ragvi.generate <questions.jsonl> <units_path> <out.jsonl> --config C3-E3-R0 \\
        --retrieval <retrieval.jsonl> --method dense[c3] --model Qwen/Qwen3-8B
"""
import argparse
import json
import re
import time
from pathlib import Path

from ragvi.chunking import _offsets
from ragvi.metrics import em_f1, normalize_answer
from ragvi.rerank import load_units
from ragvi.retrieval import DEFAULT_MODEL, load_jsonl, open_append, read_rows

REFUSAL_PHRASE = 'Không tìm thấy thông tin'
CTX_LIMIT = 1024     # mỗi đoạn đưa vào prompt cắt ở 1.024 token tham chiếu (D7)
ORACLE_LIMIT = 4000  # điều luật vàng ở điều kiện oracle cắt ở 4.000 token tham chiếu (D5)
TOP_N = 5

PROMPT_CTX = (
    'Chỉ dùng thông tin trong phần Ngữ cảnh để trả lời câu hỏi. Nếu ngữ cảnh không đủ thông tin, '
    'hãy trả lời đúng một câu: "Không tìm thấy thông tin". Trả lời ngắn gọn bằng tiếng Việt.\n\n'
    'Ngữ cảnh:\n{context}\n\n'
    'Câu hỏi: {question}'
)

PROMPT_CB = (
    'Hãy trả lời câu hỏi dưới đây. Nếu không biết chắc chắn câu trả lời, '
    'hãy trả lời đúng một câu: "Không tìm thấy thông tin". Trả lời ngắn gọn bằng tiếng Việt.\n\n'
    'Câu hỏi: {question}'
)

REFUSAL_PATTERNS = (
    'không tìm thấy thông tin',
    'không có thông tin',
    'không đủ thông tin',
    'ngữ cảnh không cung cấp',
    'ngữ cảnh không đề cập',
    'không được đề cập trong ngữ cảnh',
)


def truncate_tokens(text, tok, limit):
    """Cắt `text` ở `limit` token tham chiếu; trả về `(văn_bản_đã_cắt, có_bị_cắt_không)`."""
    if tok is None or limit is None or not text:
        return text, False
    off, _ = _offsets(text, tok)
    if len(off) <= limit:
        return text, False
    return text[:off[limit - 1][1]], True


def build_prompt(question, passages=None, tok=None, limit=CTX_LIMIT):
    """Dựng prompt cố định (closed-book khi `passages is None`) và đếm số đoạn bị cắt token."""
    if passages is None:
        return PROMPT_CB.format(question=question), 0
    blocks, n_trunc = [], 0
    for i, p in enumerate(passages, 1):
        cut, was_trunc = truncate_tokens(p, tok, limit)
        n_trunc += int(was_trunc)
        blocks.append(f'[{i}] {cut}')
    return PROMPT_CTX.format(context='\n'.join(blocks), question=question), n_trunc


def clean_generation(text):
    """Bỏ khối `<think>...</think>` (nếu có) và khoảng trắng thừa ở hai đầu."""
    text = re.sub(r'<think>.*?</think>', '', text or '', flags=re.S)
    return text.strip()


def is_refusal(answer):
    """Nhận diện câu từ chối trả lời: chuẩn hóa chữ thường/dấu câu và khớp cụm từ từ chối."""
    norm = normalize_answer(clean_generation(answer))
    if not norm:
        return True
    return any(pat in norm for pat in REFUSAL_PATTERNS)


def contains_gold_answer(answer, answers):
    """Kiểm tra câu trả lời có chứa nguyên chuỗi chuẩn hóa của ít nhất một đáp án vàng hay không (Ngày 8)."""
    if not answers:
        return None
    norm = f' {normalize_answer(answer)} '
    return any(f' {normalize_answer(a["text"] if isinstance(a, dict) else a)} ' in norm
               for a in answers if normalize_answer(a['text'] if isinstance(a, dict) else a))


def load_generator(model_name, max_new_tokens=256, load_in_4bit=True, max_input_tokens=None):
    """Nạp mô hình sinh cục bộ (4-bit NF4, temperature=0 / do_sample=False) và trả về hàm `generate(prompt)`."""
    import os
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(model_name)
    kwargs = {'device_map': 'auto'}
    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        kwargs['torch_dtype'] = compute_dtype
        if max_input_tokens is None:
            max_input_tokens = 4096 if vram_gb >= 20 else 2048
        if load_in_4bit:
            kwargs['quantization_config'] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_quant_type='nf4')
    else:
        max_input_tokens = max_input_tokens or 2048
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    model.eval()

    def generate(prompt):
        messages = [{'role': 'user', 'content': prompt}]
        if hasattr(tok, 'apply_chat_template') and getattr(tok, 'chat_template', None):
            try:
                rendered = tok.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            except TypeError:
                rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            rendered = prompt
        inputs = tok(rendered, return_tensors='pt', truncation=True, max_length=max_input_tokens).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tok.eos_token_id)
        new_tokens = out[0][inputs['input_ids'].shape[1]:].cpu()
        del inputs, out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return clean_generation(tok.decode(new_tokens, skip_special_tokens=True))

    return generate


def select_questions(questions, limit=None):
    """Chọn `limit` câu hỏi; riêng `limit=50` trên tập có cả hai loại (như dev ViQuAD) lấy đúng 35 trả lời được + 15 không (Ngày 8)."""
    if not limit or limit >= len(questions):
        return list(questions)
    ans = [q for q in questions if q.get('answerable')]
    unans = [q for q in questions if not q.get('answerable')]
    if limit == 50 and len(ans) >= 35 and len(unans) >= 15:
        return ans[:35] + unans[:15]
    return list(questions[:limit])


def run_generate(config, questions, contexts_of, out_path, generate_fn, tok=None, limit=CTX_LIMIT, model_name=''):
    """Chạy sinh câu trả lời cho danh sách `questions`, ghi từng dòng ngay vào `out_path` và bỏ qua cặp đã có.

    `contexts_of(q)` có thể trả về `None` (closed-book), `[text, ...]`, hoặc `([cid, ...], [text, ...])`.
    """
    done = {(r['config'], r['qid']) for r in read_rows(out_path)}
    written = 0
    with open_append(out_path) as f:
        for q in questions:
            qid = q['qid']
            if (config, qid) in done:
                continue
            ctx = contexts_of(q)
            if isinstance(ctx, tuple):
                cids, passages = ctx
            else:
                cids, passages = [], ctx
            prompt, n_trunc = build_prompt(q['question'], passages, tok=tok, limit=limit)
            t0 = time.perf_counter()
            answer = clean_generation(generate_fn(prompt))
            seconds = time.perf_counter() - t0
            refused = is_refusal(answer)
            answers = q.get('answers') or []
            scores = em_f1(answer, answers) if answers else {'em': None, 'f1': None}
            row = {
                'config': config, 'model': model_name, 'qid': qid, 'answerable': q['answerable'],
                'answer': answer, 'refused': refused,
                'contains_gold': contains_gold_answer(answer, answers),
                **scores,
                'context_ids': cids,
                'n_passages': 0 if passages is None else len(passages),
                'truncated_passages': n_trunc,
                'seconds': round(seconds, 3),
            }
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            f.flush()
            written += 1
    return written


def summarize_generation(rows, config=None, total_runs=6400):
    """Tổng hợp tỷ lệ từ chối đúng, fabrication rate, false refusal, EM/F1 và ước lượng giờ GPU cho 6.400 lượt."""
    own = [r for r in rows if config is None or r.get('config') == config]
    if not own:
        return {}
    ans = [r for r in own if r.get('answerable')]
    unans = [r for r in own if not r.get('answerable')]
    with_em = [r for r in ans if r.get('em') is not None]
    with_cg = [r for r in ans if r.get('contains_gold') is not None]
    total_sec = sum(r.get('seconds') or 0.0 for r in own)
    mean_sec = total_sec / len(own) if own else 0.0
    return {
        'n': len(own), 'n_ans': len(ans), 'n_unans': len(unans),
        'correct_refusal': sum(bool(r['refused']) for r in unans) / len(unans) if unans else None,
        'fabrication': sum(not r['refused'] for r in unans) / len(unans) if unans else None,
        'false_refusal': sum(bool(r['refused']) for r in ans) / len(ans) if ans else None,
        'contains_gold': sum(bool(r['contains_gold']) for r in with_cg) / len(with_cg) if with_cg else None,
        'em': sum(r['em'] for r in with_em) / len(with_em) if with_em else None,
        'f1': sum(r['f1'] for r in with_em) / len(with_em) if with_em else None,
        'truncated_passages': sum(r.get('truncated_passages', 0) for r in own),
        'q_per_min': 60.0 / mean_sec if mean_sec > 0 else None,
        'est_gpu_hours_6400': (mean_sec * total_runs) / 3600.0 if mean_sec > 0 else None,
    }


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument('questions_path', help='dev.jsonl hoặc test.jsonl')
    parser.add_argument('units_path', help='thư mục zalo_legal, corpus.jsonl ViQuAD, hoặc chunks/*.jsonl')
    parser.add_argument('out_path', help='file JSONL lưu câu trả lời')
    parser.add_argument('--config', required=True, help='closed_book, oracle, hoặc nhãn cấu hình như C3-E3-R0')
    parser.add_argument('--retrieval', help='file retrieval_*.jsonl (bắt buộc khi config khác closed_book/oracle)')
    parser.add_argument('--method', help='nhãn phương pháp trong file retrieval (ví dụ dense[c3], hybrid[c3][word])')
    parser.add_argument('--model', default='Qwen/Qwen3-8B')
    parser.add_argument('--ref-tokenizer', default=DEFAULT_MODEL)
    parser.add_argument('--limit', type=int, help='chỉ chạy trên N câu đầu (ví dụ 50 câu dev ở Ngày 8–9)')
    parser.add_argument('--top-n', type=int, default=TOP_N)
    args = parser.parse_args(argv)

    questions = select_questions(load_jsonl(args.questions_path), args.limit)
    text_of, _, _ = load_units(args.units_path)

    if args.config == 'closed_book':
        contexts_of = lambda q: None
        tok_limit = CTX_LIMIT
    elif args.config == 'oracle':
        # Oracle chỉ tính trên câu trả lời được (D5), trừ khi thử dev ViQuAD có đoạn gốc (Ngày 8).
        def contexts_of(q):
            ids = [d for d in (q.get('gold_doc_ids') or ([q['gold_doc_id']] if 'gold_doc_id' in q else []))
                   if d in text_of]
            return ids, [text_of[d] for d in ids]
        tok_limit = ORACLE_LIMIT
    else:
        if not args.retrieval or not args.method:
            raise SystemExit('Cấu hình RAG cần cả --retrieval và --method.')
        top_map = {r['qid']: r['top'] for r in read_rows(args.retrieval)
                   if r.get('method') == args.method and 'top' in r}
        missing = [q['qid'] for q in questions if q['qid'] not in top_map]
        if missing:
            raise SystemExit(f'Thiếu {len(missing)} câu hỏi cho method={args.method!r} trong {args.retrieval}.')

        def contexts_of(q):
            ids = top_map[q['qid']][:args.top_n]
            return ids, [text_of[cid] for cid in ids]
        tok_limit = CTX_LIMIT

    done = {(r['config'], r['qid']) for r in read_rows(args.out_path)}
    need = any((args.config, q['qid']) not in done for q in questions)
    if need:
        from transformers import AutoTokenizer
        ref_tok = None if args.config == 'closed_book' else AutoTokenizer.from_pretrained(args.ref_tokenizer)
        gen_fn = load_generator(args.model)
    else:
        ref_tok, gen_fn = None, (lambda p: '')

    written = run_generate(args.config, questions, contexts_of, args.out_path, gen_fn,
                           tok=ref_tok, limit=tok_limit, model_name=args.model)
    summary = summarize_generation(read_rows(args.out_path), config=args.config)
    print(f'Ghi thêm {written} câu trả lời ({args.config}, model={args.model}) vào {args.out_path}')
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == '__main__':
    main()


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/judge.py`

Chép mã nguồn mới nhất (có checkpoint/resume embedding + fp16/bf16) vào `/content/src/ragvi/`.

In [ ]:
%%writefile /content/src/ragvi/judge.py
"""LLM-as-judge tiếng Việt, chỉ số "nhầm điều luật" (H1b), phiếu gán nhãn người 100 câu dev và Cohen's κ (Ngày 11–13).

Chạy từ thư mục gốc của kho mã:
    PYTHONPATH=src python -m ragvi.judge <answers.jsonl> <questions.jsonl> <corpus> <out_judge.jsonl> --model ...
    PYTHONPATH=src python -m ragvi.judge --human-sheet <answers.jsonl> <questions.jsonl> <corpus> <out_sheet.md>
    PYTHONPATH=src python -m ragvi.judge --kappa <graded_sheet.md> <out_judge.jsonl>
"""
import argparse
import hashlib
import json
import random
import re
import time
from pathlib import Path

from ragvi.generate import clean_generation, is_refusal, load_generator
from ragvi.rerank import load_units
from ragvi.retrieval import load_jsonl, open_append, read_rows

ARTICLE_RE = re.compile(r'\bđiều\s+(\d+)\b', re.I)


def blind_id(config, qid):
    """Mã ẩn danh 10 ký tự hex cho cặp (config, qid) để người gán nhãn không biết tên điều kiện (Ngày 12)."""
    return hashlib.sha256(f'{config}::{qid}'.encode('utf-8')).hexdigest()[:10]

PROMPT_JUDGE = """Bạn là giám khảo đánh giá câu trả lời của hệ thống hỏi đáp tiếng Việt.
Hãy đối chiếu "Câu trả lời" với "Ngữ cảnh đã cung cấp" và "Bằng chứng vàng", rồi trả về ĐÚNG MỘT đối tượng JSON hợp lệ (không giải thích thêm ngoài JSON) theo mẫu:
{{"claims": [{{"text": "<mệnh đề trong câu trả lời>", "supported_by_context": true, "supported_by_gold": false}}], "correctness": 1}}

Quy tắc chấm:
1. Tách "Câu trả lời" thành các mệnh đề khẳng định cụ thể trong danh sách `claims`.
2. Với mỗi mệnh đề:
   - `supported_by_context`: `true` nếu mệnh đề được "Ngữ cảnh đã cung cấp" hỗ trợ hoàn toàn, `false` nếu bịa đặt hoặc mâu thuẫn với ngữ cảnh (nếu Ngữ cảnh là "(không cung cấp - closed-book)" thì để `null`).
   - `supported_by_gold`: `true` nếu mệnh đề được "Bằng chứng vàng" hỗ trợ hoàn toàn, `false` nếu có thông tin sai hoặc không có căn cứ trong bằng chứng vàng.
3. `correctness` (điểm độ đúng của câu trả lời so với Bằng chứng vàng):
   - `0`: Sai, lạc đề, hoặc không trả lời được câu hỏi.
   - `1`: Đúng một phần nhưng còn thiếu ý quan trọng hoặc diễn đạt chưa chính xác hoàn toàn.
   - `2`: Đúng và đầy đủ theo Bằng chứng vàng.

Câu hỏi: {question}

Ngữ cảnh đã cung cấp:
{context}

Bằng chứng vàng:
{gold}

Câu trả lời:
{answer}"""


def cited_articles(answer):
    """Danh sách số Điều được viện dẫn trong `answer` (theo thứ tự xuất hiện, không trùng)."""
    seen, out = set(), []
    for num in ARTICLE_RE.findall(answer or ''):
        num = str(int(num))
        if num not in seen:
            seen.add(num)
            out.append(num)
    return out


def gold_articles(gold_doc_ids):
    """Tập số Điều của các điều luật vàng (phần sau dấu `+` trong mã tài liệu Zalo; D2)."""
    return {str(int(d.rsplit('+', 1)[1])) for d in (gold_doc_ids or []) if '+' in d and d.rsplit('+', 1)[1].isdigit()}


def wrong_article_citation(answer, gold_doc_ids):
    """Chỉ số 'nhầm điều luật' cho H1b (D2): câu trả lời dẫn số Điều không khớp số điều của điều luật vàng."""
    cited = cited_articles(answer)
    gold = gold_articles(gold_doc_ids)
    return {
        'cited_articles': cited,
        'gold_articles': sorted(gold),
        'wrong_article': bool(cited) and any(a not in gold for a in cited),
    }


def parse_judge_json(raw):
    """Trích và kiểm tra đối tượng JSON `{"claims": [...], "correctness": 0|1|2}` từ đầu ra của judge."""
    text = clean_generation(raw)
    m = re.search(r'\{.*\}', text, flags=re.S)
    if not m:
        raise ValueError(f'Không tìm thấy đối tượng JSON trong đầu ra judge: {text[:120]!r}')
    data = json.loads(m.group(0))
    if not isinstance(data, dict) or not isinstance(data.get('claims'), list):
        raise ValueError(f'JSON thiếu danh sách claims hợp lệ: {data!r}')
    if data.get('correctness') not in (0, 1, 2):
        raise ValueError(f'Trường correctness phải thuộc {{0, 1, 2}}, nhận được: {data.get("correctness")!r}')
    for c in data['claims']:
        if not isinstance(c, dict) or 'text' not in c or 'supported_by_gold' not in c:
            raise ValueError(f'Mệnh đề trong claims không hợp lệ: {c!r}')
    return data


def derive_judge_metrics(parsed, refused, is_closed_book=False):
    """Từ JSON của judge tính H_gold, H_ctx và correctness theo quyết định D3–D4.

    Câu từ chối không có mệnh đề nào, tính vào false refusal chứ không tính vào H_gold hay H_ctx.
    """
    if refused:
        return {'correctness': 0, 'h_gold': None, 'h_ctx': None, 'n_claims': 0}
    claims = parsed.get('claims') or []
    h_gold = int(any(not bool(c.get('supported_by_gold')) for c in claims))
    h_ctx = None if is_closed_book else int(any(not bool(c.get('supported_by_context')) for c in claims))
    return {'correctness': int(parsed['correctness']), 'h_gold': h_gold, 'h_ctx': h_ctx, 'n_claims': len(claims)}


def format_gold_evidence(question_row, corpus_text_of):
    """Ghép bằng chứng vàng (đoạn vàng + đáp án vàng với ViQuAD; điều luật vàng với Zalo) cho judge."""
    ids = question_row.get('gold_doc_ids') or ([question_row['gold_doc_id']] if 'gold_doc_id' in question_row else [])
    docs = [corpus_text_of[d] for d in ids if d in corpus_text_of]
    answers = [a['text'] if isinstance(a, dict) else a for a in (question_row.get('answers') or [])]
    parts = []
    if answers:
        parts.append('Đáp án vàng: ' + ' | '.join(answers))
    parts.extend(docs)
    return '\n\n'.join(parts)


JUDGE_MAX_CHARS = 3000  # giới hạn mỗi phần ngữ cảnh/bằng chứng vàng (~750 token) để vừa VRAM T4 (15 GB)


def _clip(text, max_chars=JUDGE_MAX_CHARS):
    return text if len(text) <= max_chars else text[:max_chars].rstrip() + ' …'


def run_judge(answer_rows, questions_by_qid, context_of, gold_of, out_path, judge_fn, model_name=''):
    """Chấm điểm các câu trả lời của câu hỏi trả lời được (`answerable=True`), ghi nối tiếp vào `out_path`."""
    done = {(r['config'], r['qid']) for r in read_rows(out_path) if not r.get('error')}
    written = 0
    with open_append(out_path) as f:
        for row in answer_rows:
            if not row.get('answerable', True):
                continue
            config, qid = row['config'], row['qid']
            if (config, qid) in done or qid not in questions_by_qid:
                continue
            q = questions_by_qid[qid]
            refused = bool(row.get('refused') or is_refusal(row.get('answer', '')))
            is_cb = config == 'closed_book'
            cite = wrong_article_citation(row.get('answer', ''), q.get('gold_doc_ids')) if 'gold_doc_ids' in q else {}
            if refused:
                scores = derive_judge_metrics({'claims': [], 'correctness': 0}, True, is_closed_book=is_cb)
                out_row = {'config': config, 'qid': qid, 'judge_model': model_name, 'refused': True,
                           'claims': [], **scores, **cite, 'error': None}
            else:
                ctx_text = '(không cung cấp - closed-book)' if is_cb else _clip(context_of(config, qid))
                prompt = PROMPT_JUDGE.format(
                    question=q['question'], context=ctx_text, gold=_clip(gold_of(q)), answer=row.get('answer', ''))
                t0 = time.perf_counter()
                raw = judge_fn(prompt)
                elapsed = round(time.perf_counter() - t0, 3)
                try:
                    parsed = parse_judge_json(raw)
                    scores = derive_judge_metrics(parsed, False, is_closed_book=is_cb)
                    out_row = {'config': config, 'qid': qid, 'judge_model': model_name, 'refused': False,
                               'claims': parsed['claims'], **scores, **cite, 'seconds': elapsed, 'error': None}
                except (ValueError, json.JSONDecodeError) as e:
                    out_row = {'config': config, 'qid': qid, 'judge_model': model_name, 'refused': False,
                               'claims': None, 'correctness': None, 'h_gold': None, 'h_ctx': None, 'n_claims': None,
                               **cite, 'seconds': elapsed, 'raw': raw, 'error': str(e)}
            f.write(json.dumps(out_row, ensure_ascii=False) + '\n')
            f.flush()
            written += 1
    return written


def build_human_sheet(answer_rows, questions_by_qid, context_of, gold_of, n=50, seed=0):
    """Trộn ngẫu nhiên `n` câu trả lời dev (ẩn tên cấu hình) để người gán nhãn chấm trước khi xem điểm judge."""
    pool = [r for r in answer_rows if r.get('answerable', True) and not r.get('refused')
            and r['qid'] in questions_by_qid]
    pool.sort(key=lambda r: (r['config'], r['qid']))
    picked = random.Random(seed).sample(pool, min(n, len(pool)))
    lines = [
        '# Phiếu gán nhãn tay đối chiếu Judge (Ngày 12–13)\n\n'
        'Hướng dẫn: với mỗi mẫu (đã ẩn tên cấu hình), đọc Câu hỏi, Ngữ cảnh đã cung cấp, Bằng chứng vàng và '
        'Câu trả lời rồi đánh `[x]` vào đúng một ô cho mỗi dòng:\n'
        '- `Correctness` (so với Bằng chứng vàng): `0` (sai), `1` (đúng một phần), `2` (đúng và đầy đủ).\n'
        '- `H_gold`: Câu trả lời có mệnh đề nào KHÔNG được Bằng chứng vàng hỗ trợ không? (`0` = không, `1` = có).\n'
        '- `H_ctx`: Câu trả lời có mệnh đề nào KHÔNG được Ngữ cảnh hỗ trợ không? (`0` = không, `1` = có, `NA` = không có ngữ cảnh).\n'
    ]
    for i, r in enumerate(picked, 1):
        config, qid = r['config'], r['qid']
        q = questions_by_qid[qid]
        ctx = '(không cung cấp - closed-book)' if config == 'closed_book' else context_of(config, qid)
        bid = blind_id(config, qid)
        lines.append(
            f'\n---\n\n## Mẫu {i} — `{bid}`\n\n'
            f'**Câu hỏi:** {q["question"]}\n\n'
            f'**Ngữ cảnh đã cung cấp:**\n> ' + '\n> '.join(ctx.splitlines()) + '\n\n'
            f'**Bằng chứng vàng:**\n> ' + '\n> '.join(gold_of(q).splitlines()) + '\n\n'
            f'**Câu trả lời:** {r["answer"]}\n\n'
            'Correctness: [ ] 0   [ ] 1   [ ] 2\n'
            'H_gold: [ ] 0   [ ] 1\n'
            'H_ctx: [ ] 0   [ ] 1   [ ] NA\n'
        )
    return ''.join(lines)


def cohen_kappa(y1, y2, weights=None):
    """Cohen's κ (không trọng số hoặc trọng số tuyến tính `weights='linear'`) cho hai danh sách nhãn nguyên."""
    if len(y1) != len(y2) or not y1:
        raise ValueError('Hai danh sách nhãn phải cùng độ dài và khác rỗng.')
    labels = sorted(set(y1) | set(y2))
    if len(labels) == 1:
        return 1.0
    n = len(y1)
    idx = {lab: i for i, lab in enumerate(labels)}
    k = len(labels)
    mat = [[0] * k for _ in range(k)]
    for a, b in zip(y1, y2):
        mat[idx[a]][idx[b]] += 1
    row_sum = [sum(mat[i]) for i in range(k)]
    col_sum = [sum(mat[i][j] for i in range(k)) for j in range(k)]
    max_span = max(labels[-1] - labels[0], 1)

    def w(i, j):
        if weights == 'linear':
            return abs(labels[i] - labels[j]) / max_span
        return float(i != j)

    po_dis = sum(w(i, j) * mat[i][j] for i in range(k) for j in range(k)) / n
    pe_dis = sum(w(i, j) * row_sum[i] * col_sum[j] for i in range(k) for j in range(k)) / (n * n)
    return 1.0 if pe_dis == 0 else 1.0 - po_dis / pe_dis


def parse_human_sheet(markdown):
    """Đọc các nhãn đã đánh `[x]` từ phiếu `build_human_sheet` (khóa là mã ẩn danh `blind_id` hoặc `config::qid`)."""
    out = {}
    for block in re.split(r'^## Mẫu \d+ — ', markdown, flags=re.M)[1:]:
        key = block.split('`')[1]
        if '::' in key:
            key = blind_id(*key.split('::', 1))
        c_line = next((l for l in block.splitlines() if l.startswith('Correctness:')), '')
        g_line = next((l for l in block.splitlines() if l.startswith('H_gold:')), '')
        x_line = next((l for l in block.splitlines() if l.startswith('H_ctx:')), '')
        c_hits = [int(m) for m in re.findall(r'\[[xX]\]\s*([012])', c_line)]
        g_hits = [int(m) for m in re.findall(r'\[[xX]\]\s*([01])', g_line)]
        x_hits = re.findall(r'\[[xX]\]\s*(0|1|NA)', x_line)
        if len(c_hits) == 1 and len(g_hits) == 1 and len(x_hits) == 1:
            out[key] = {
                'correctness': c_hits[0],
                'h_gold': g_hits[0],
                'h_ctx': None if x_hits[0] == 'NA' else int(x_hits[0]),
            }
    return out


def evaluate_kappa(human_labels, judge_rows):
    """Tính Cohen's κ giữa người và judge cho H_gold, H_ctx và correctness (có trọng số tuyến tính)."""
    norm_human = {(blind_id(*k) if isinstance(k, tuple) else k): v for k, v in human_labels.items()}
    judge_map = {blind_id(r['config'], r['qid']): r
                 for r in judge_rows if r.get('error') is None and not r.get('refused')}
    shared = sorted(set(norm_human) & set(judge_map))
    if not shared:
        raise ValueError('Không có cặp (config, qid) chung nào giữa phiếu nhãn người và kết quả judge.')
    h_gold_h = [norm_human[k]['h_gold'] for k in shared if judge_map[k].get('h_gold') is not None]
    h_gold_j = [judge_map[k]['h_gold'] for k in shared if judge_map[k].get('h_gold') is not None]
    ctx_keys = [k for k in shared if norm_human[k]['h_ctx'] is not None and judge_map[k].get('h_ctx') is not None]
    corr_h = [norm_human[k]['correctness'] for k in shared if judge_map[k].get('correctness') is not None]
    corr_j = [judge_map[k]['correctness'] for k in shared if judge_map[k].get('correctness') is not None]
    return {
        'n_shared': len(shared),
        'kappa_h_gold': cohen_kappa(h_gold_h, h_gold_j),
        'kappa_h_ctx': cohen_kappa([norm_human[k]['h_ctx'] for k in ctx_keys],
                                   [judge_map[k]['h_ctx'] for k in ctx_keys]) if ctx_keys else None,
        'kappa_correctness_linear': cohen_kappa(corr_h, corr_j, weights='linear'),
    }


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument('paths', nargs='*', help='<answers.jsonl> <questions.jsonl> <corpus> <out.jsonl/.md>')
    parser.add_argument('--chunks', help='file chunks/*.jsonl (ví dụ __c3.jsonl) để lấy văn bản các chunk ngữ cảnh')
    parser.add_argument('--model', default='Qwen/Qwen2.5-7B-Instruct')
    parser.add_argument('--limit', type=int, help='chỉ chấm N câu trả lời được đầu tiên (ví dụ 20 câu dev ở Ngày 11)')
    parser.add_argument('--human-sheet', action='store_true', help='xuất phiếu gán nhãn tay (Ngày 12)')
    parser.add_argument('--kappa', nargs=2, metavar=('PHIEU_DA_CHAM', 'JUDGE_JSONL'), help='tính κ (Ngày 13)')
    parser.add_argument('--n', type=int, default=50)
    parser.add_argument('--seed', type=int, default=0)
    args = parser.parse_args(argv)

    if args.kappa:
        sheet_path, judge_path = args.kappa
        human = parse_human_sheet(Path(sheet_path).read_text(encoding='utf-8'))
        res = evaluate_kappa(human, read_rows(judge_path))
        print(json.dumps(res, ensure_ascii=False, indent=2))
        return

    if len(args.paths) != 4:
        parser.error('Cần 4 đường dẫn: <answers.jsonl> <questions.jsonl> <corpus> <out>, hoặc dùng --kappa')
    ans_path, q_path, corpus_path, out_path = args.paths
    answers = [r for r in read_rows(ans_path) if r.get('answerable', True)]
    if args.limit:
        answers = answers[:args.limit]
    questions = {q['qid']: q for q in load_jsonl(q_path)}
    corpus_text_of, _, _ = load_units(corpus_path)
    unit_text_of = dict(corpus_text_of)
    if args.chunks and Path(args.chunks).exists():
        chunk_text_of, _, _ = load_units(args.chunks)
        unit_text_of.update(chunk_text_of)
    ctx_ids_of = {(r['config'], r['qid']): r.get('context_ids') or [] for r in answers}
    gold_of = lambda q: format_gold_evidence(q, corpus_text_of)

    def context_of(cfg, qid):
        cids = ctx_ids_of.get((cfg, qid)) or []
        passages = [unit_text_of[c] for c in cids if c in unit_text_of]
        if passages:
            return '\n'.join(f'[{i}] {p}' for i, p in enumerate(passages, 1))
        return gold_of(questions[qid])

    if args.human_sheet:
        if Path(out_path).exists():
            raise SystemExit(f'{out_path} đã tồn tại; không ghi đè phiếu gán nhãn.')
        md = build_human_sheet(answers, questions, context_of, gold_of, n=args.n, seed=args.seed)
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        Path(out_path).write_text(md, encoding='utf-8')
        print(f'Đã ghi phiếu gán nhãn vào {out_path}')
        return

    done = {(r['config'], r['qid']) for r in read_rows(out_path) if not r.get('error')}
    need = any(r.get('answerable', True) and (r['config'], r['qid']) not in done for r in answers)
    judge_fn = load_generator(args.model) if need else (lambda p: '{}')
    written = run_judge(answers, questions, context_of, gold_of, out_path, judge_fn, model_name=args.model)
    rows = [r for r in read_rows(out_path) if r.get('error') is None]
    errors = sum(r.get('error') is not None for r in read_rows(out_path))
    print(f'Đã chấm thêm {written} câu trả lời vào {out_path} (hợp lệ: {len(rows)}, lỗi JSON: {errors})')


if __name__ == '__main__':
    main()


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/metrics.py`

Chép mã nguồn mới nhất (có checkpoint/resume embedding + fp16/bf16) vào `/content/src/ragvi/`.

In [ ]:
%%writefile /content/src/ragvi/metrics.py
"""Chỉ số truy xuất (Recall@5, Recall@20, MRR@10, nDCG@5) và độ đúng ViQuAD (EM, token-F1 theo âm tiết; D2)."""
from collections import Counter
import math
import re

METRIC_KEYS = ('r5', 'r20', 'mrr10', 'ndcg5')


def normalize_answer(text):
    """Chuẩn hóa câu trả lời theo âm tiết: chữ thường, bỏ dấu câu, gộp khoảng trắng."""
    return ' '.join(re.findall(r'\w+', (text or '').lower()))


def em_f1(prediction, gold_texts):
    """EM và token-F1 theo âm tiết trên tập đáp án vàng (lấy giá trị lớn nhất qua các đáp án vàng; D2)."""
    if not gold_texts:
        return {'em': None, 'f1': None}
    pred_norm = normalize_answer(prediction)
    pred_toks = pred_norm.split()
    best_em, best_f1 = 0.0, 0.0
    for gold in gold_texts:
        gold_norm = normalize_answer(gold['text'] if isinstance(gold, dict) else gold)
        gold_toks = gold_norm.split()
        best_em = max(best_em, float(pred_norm == gold_norm))
        if not pred_toks or not gold_toks:
            best_f1 = max(best_f1, float(pred_toks == gold_toks))
            continue
        common = sum((Counter(pred_toks) & Counter(gold_toks)).values())
        if common:
            best_f1 = max(best_f1, 2 * common / (len(pred_toks) + len(gold_toks)))
    return {'em': best_em, 'f1': best_f1}


def metrics(hits, n_rel):
    """hits: danh sách True/False theo thứ hạng, mỗi tài liệu vàng chỉ True một lần."""
    dcg = sum(h / math.log2(i + 2) for i, h in enumerate(hits[:5]))
    idcg = sum(1 / math.log2(i + 2) for i in range(min(n_rel, 5)))
    return {'r5': sum(hits[:5]) / n_rel, 'r20': sum(hits[:20]) / n_rel,
            'mrr10': next((1 / (i + 1) for i, h in enumerate(hits[:10]) if h), 0.0),
            'ndcg5': dcg / idcg}


def zalo_hits(ranked_doc_ids, gold):
    """Mỗi điều luật vàng chỉ tính một lần, ở lần xuất hiện đầu tiên trong danh sách xếp hạng."""
    seen, hits = set(), []
    for d in ranked_doc_ids:
        hits.append(d in gold and d not in seen)
        seen.add(d)
    return hits


def viquad_hits(ranked_chunks, gold_doc_id, answers, min_cover=0.8):
    """Tiêu chí D2 cho ViQuAD; ranked_chunks là [(doc_id, start, end)] theo thứ hạng.

    Chunk đúng khi thuộc gold_doc_id và phủ ít nhất `min_cover` số ký tự của ít nhất một đáp án vàng
    (answers: [{'start', 'end', ...}] như trong tập đánh giá). n_rel = 1: chỉ lần trúng đầu tiên là True.
    """
    found, hits = False, []
    for doc_id, start, end in ranked_chunks:
        ok = not found and doc_id == gold_doc_id and any(
            min(end, a['end']) - max(start, a['start']) >= min_cover * (a['end'] - a['start']) for a in answers)
        hits.append(ok)
        found |= ok
    return hits


def mean_metrics(rows):
    """Trung bình từng chỉ số trên nhiều truy vấn; rows là các dict có đủ khóa METRIC_KEYS."""
    rows = list(rows)
    if not rows:
        raise ValueError('Không có truy vấn nào để lấy trung bình.')
    return {k: sum(r[k] for r in rows) / len(rows) for k in METRIC_KEYS}


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/rerank.py`

Chép mã nguồn mới nhất (có checkpoint/resume embedding + fp16/bf16) vào `/content/src/ragvi/`.

In [ ]:
%%writefile /content/src/ragvi/rerank.py
"""Xếp hạng lại (reranking R1: BAAI/bge-reranker-v2-m3) trên top-20 ứng viên truy xuất.

Chạy từ thư mục gốc của kho mã:
    PYTHONPATH=src python -m ragvi.rerank <retrieval.jsonl> <questions.jsonl> <kho_hoặc_chunks> <out.jsonl> [--method hybrid[word]]
"""
import argparse
import json
import time
from pathlib import Path

from ragvi.metrics import METRIC_KEYS, mean_metrics, metrics, viquad_hits, zalo_hits
from ragvi.retrieval import format_table, load_jsonl, open_append, read_rows, summarize, top_k

DEFAULT_RERANKER = 'BAAI/bge-reranker-v2-m3'


def rerank_top(query, candidate_ids, text_of, scorer):
    """Chấm điểm lại danh sách `candidate_ids` bằng `scorer(query, passages)`, giữ nguyên tập ứng viên (Recall@20 bất biến)."""
    if not candidate_ids:
        return []
    passages = [text_of[cid] for cid in candidate_ids]
    scores = scorer(query, passages)
    return [candidate_ids[i] for i in top_k(scores, len(candidate_ids))]


def load_reranker(model_name=DEFAULT_RERANKER, max_length=512, batch_size=32):
    """Nạp CrossEncoder và trả về hàm `scorer(query, passages) -> danh sách điểm`."""
    from sentence_transformers import CrossEncoder

    model = CrossEncoder(model_name, max_length=max_length)
    return lambda query, passages: model.predict(
        [(query, p) for p in passages], batch_size=batch_size, show_progress_bar=False)


def run_rerank(retrieval_rows, questions, gold, text_of, out_path, scorer, method, out_method=None, hit_fn=None):
    """Xếp hạng lại kết quả của `method` trong `retrieval_rows`, ghi nối tiếp vào `out_path`."""
    out_method = out_method or f'{method}+r1'
    candidates = {r['qid']: r['top'] for r in retrieval_rows if r.get('method') == method and 'top' in r}
    done = {(r['method'], r['qid']) for r in read_rows(out_path)}
    written = 0
    with open_append(out_path) as f:
        for qid, question in questions.items():
            if (out_method, qid) in done or qid not in candidates:
                continue
            t0 = time.perf_counter()
            top = rerank_top(question, candidates[qid], text_of, scorer)
            elapsed = time.perf_counter() - t0
            if gold[qid] is None:
                scores = dict.fromkeys(METRIC_KEYS)
            elif hit_fn is not None:
                scores = hit_fn(qid, top)
            else:
                scores = metrics(zalo_hits(top, gold[qid]), len(gold[qid]))
            row = {'method': out_method, 'qid': qid, **scores, 'top': top, 'seconds': round(elapsed, 4)}
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            f.flush()
            written += 1
    return written


def load_units(path):
    """Đọc văn bản và ánh xạ (parent_of, span_of) từ kho Zalo, corpus.jsonl ViQuAD, hoặc file đệm chunks/*.jsonl."""
    p = Path(path)
    if p.is_file() and p.name.endswith('.jsonl'):
        rows = load_jsonl(p)
        if rows and isinstance(rows[0], list):  # [chunk_id, doc_id, start, end, text]
            return ({r[0]: r[4] for r in rows},
                    {r[0]: r[1] for r in rows},
                    {r[0]: (r[1], r[2], r[3]) for r in rows})
        return ({r['doc_id']: r['text'] for r in rows},
                {r['doc_id']: r['doc_id'] for r in rows},
                {r['doc_id']: (r['doc_id'], 0, len(r['text'])) for r in rows})
    from ragvi.data import load_zalo_corpus

    corpus = load_zalo_corpus(p)
    return corpus, {d: d for d in corpus}, {d: (d, 0, len(t)) for d, t in corpus.items()}


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument('retrieval_path', help='file retrieval_*.jsonl chứa top-20 ứng viên')
    parser.add_argument('questions_path', help='dev.jsonl hoặc test.jsonl')
    parser.add_argument('units_path', help='thư mục zalo_legal, corpus.jsonl ViQuAD, hoặc chunks/*.jsonl')
    parser.add_argument('out_path', help='file JSONL đầu ra (có thể trùng retrieval_path)')
    parser.add_argument('--method', default='dense', help='nhãn phương pháp đầu vào trong retrieval_path')
    parser.add_argument('--out-method', help='nhãn phương pháp đầu ra (mặc định: <method>+r1)')
    parser.add_argument('--model', default=DEFAULT_RERANKER)
    parser.add_argument('--batch-size', type=int, default=32)
    args = parser.parse_args(argv)

    q_rows = load_jsonl(args.questions_path)
    questions = {q['qid']: q['question'] for q in q_rows}
    is_viquad = any('gold_doc_id' in q for q in q_rows)
    gold = {q['qid']: (set(q['gold_doc_ids']) if 'gold_doc_ids' in q else {q['gold_doc_id']})
            if q['answerable'] else None for q in q_rows}
    answers_of = {q['qid']: q.get('answers') or [] for q in q_rows}
    text_of, parent_of, span_of = load_units(args.units_path)
    ret_rows = read_rows(args.retrieval_path)

    out_method = args.out_method or f'{args.method}+r1'
    done = {(r['method'], r['qid']) for r in read_rows(args.out_path)}
    need = any((out_method, q) not in done for q in questions)
    scorer = load_reranker(args.model, batch_size=args.batch_size) if need else (lambda q, ps: [])

    def hit_fn(qid, top):
        if is_viquad:
            return metrics(viquad_hits([span_of[c] for c in top], next(iter(gold[qid])), answers_of[qid]), 1)
        return metrics(zalo_hits([parent_of[c] for c in top], gold[qid]), len(gold[qid]))

    written = run_rerank(ret_rows, questions, gold, text_of, args.out_path, scorer,
                         args.method, out_method=out_method, hit_fn=hit_fn)
    print(f'Ghi thêm {written} dòng ({out_method}) vào {args.out_path}')
    base_table = summarize(args.retrieval_path, [args.method])
    rerank_table = summarize(args.out_path, [out_method])
    print(format_table({**base_table, **rerank_table}))


if __name__ == '__main__':
    main()


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/retrieval.py`

Chép mã nguồn mới nhất (có checkpoint/resume embedding + fp16/bf16) vào `/content/src/ragvi/`.

In [ ]:
%%writefile /content/src/ragvi/retrieval.py
"""Truy xuất BM25, dense và hybrid ở mức tài liệu: điều luật Zalo hoặc đoạn ViQuAD.

Chạy trên Colab, từ thư mục gốc của kho mã:
    PYTHONPATH=src python -m ragvi.retrieval <processed>/zalo_legal <thư mục đầu ra> --methods bm25,dense,hybrid
    PYTHONPATH=src python -m ragvi.retrieval <processed>/zalo_legal <thư mục đầu ra> --methods bm25 --tokenizer word
    PYTHONPATH=src python -m ragvi.retrieval <processed>/zalo_legal <thư mục đầu ra> --run zalo_test \\
        --questions <eval>/zalo/test.jsonl --removed <eval>/zalo/removed_doc_ids.json
    PYTHONPATH=src python -m ragvi.retrieval <eval>/viquad/corpus.jsonl <thư mục đầu ra> --run viquad_test \\
        --questions <eval>/viquad/test.jsonl
"""
import argparse
import json
import random
import re
import time
from pathlib import Path

import numpy as np

from ragvi.metrics import METRIC_KEYS, mean_metrics, metrics, viquad_hits, zalo_hits

DEFAULT_MODEL = 'BAAI/bge-m3'
# alpha nhân với dense, min–max theo từng truy vấn trên cả kho: chốt 14/11 theo Nguyen & Quan
# (Findings of EACL 2026) và mã ViRE của họ; xem Nhật ký thay đổi trong reports/quyet-dinh-thiet-ke.md.
ALPHA = 0.7
TOP_K = 20


def model_prefixes(model_name):
    """Tiền tố (query_prefix, passage_prefix); dòng multilingual-e5 yêu cầu 'query: ' và 'passage: ' (D7)."""
    return ('query: ', 'passage: ') if 'e5' in model_name.lower() else ('', '')


def cached_chunks(docs, chunking, path, titles=None, tokenizer_name=DEFAULT_MODEL):
    """Dựng hoặc đọc đệm danh sách chunk [(chunk_id, doc_id, start, end, text)] tại `path`."""
    rows = read_rows(path)
    if rows and {r[1] for r in rows} == set(docs):
        return [tuple(r) for r in rows]

    from transformers import AutoTokenizer
    from ragvi.chunking import chunk_document, chunk_fixed, chunk_sentence, chunk_struct

    tok = None if chunking == 'c4' else AutoTokenizer.from_pretrained(tokenizer_name)
    out = []
    for d, text in docs.items():
        if chunking == 'c1':
            out += chunk_fixed(d, text, tok)
        elif chunking == 'c3':
            out += chunk_sentence(d, text, tok)
        elif chunking == 'c4':
            out += chunk_document(d, text)
        elif chunking in ('c5', 'c5_nohead'):
            title = titles[d] if titles else text.split('\n', 1)[0]
            body = text[len(title) + 1:] if text.startswith(title + '\n') else ''
            out += chunk_struct(d, title, body, tok, heading=(chunking == 'c5'))
        else:
            raise ValueError(f'Cách phân đoạn không hợp lệ: {chunking}')
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(''.join(json.dumps(list(c), ensure_ascii=False) + '\n' for c in out), encoding='utf-8')
    return out


# ---------- BM25 ----------

def tokenize_syllables(text):
    """Tách theo âm tiết: chữ thường, cắt ở khoảng trắng và dấu câu."""
    return re.findall(r'\w+', text.lower())


def tokenize_words(text):
    """Tách từ bằng underthesea: chữ thường, bỏ token chỉ có dấu câu, nối âm tiết trong một từ bằng '_'."""
    from underthesea import word_tokenize

    return [re.sub(r'\s+', '_', w.lower()) for w in word_tokenize(text) if re.search(r'\w', w)]


def cached_tokens(doc_ids, texts, path, tokenize=tokenize_words):
    """Token của từng tài liệu, đệm ra JSONL `[doc_id, "tok tok ..."]` theo doc_id.

    Ghi nối tiếp từng tài liệu: lượt chạy bị ngắt thì chạy lại chỉ tách phần còn thiếu;
    kho thu hẹp dùng lại đúng các dòng của kho đầy đủ.
    """
    cache = dict(read_rows(path))
    missing = [(d, t) for d, t in zip(doc_ids, texts) if d not in cache]
    if missing:
        start = time.perf_counter()
        with open_append(path) as f:
            for d, t in missing:
                cache[d] = ' '.join(tokenize(t))
                f.write(json.dumps([d, cache[d]], ensure_ascii=False) + '\n')
        print(f'Đã tách từ {len(missing)} tài liệu trong {time.perf_counter() - start:.1f} giây, đệm tại {path}')
    return [cache[d].split() for d in doc_ids]


def build_bm25(texts, tokenize=tokenize_syllables):
    """Dựng chỉ mục BM25Okapi; đổi `tokenize` thành underthesea.word_tokenize để tách từ."""
    from rank_bm25 import BM25Okapi

    return BM25Okapi([tokenize(t) for t in texts])


def bm25_scores(index, query, tokenize=tokenize_syllables):
    """Vector điểm BM25 của một truy vấn trên cả kho; `tokenize` phải trùng với lúc dựng chỉ mục."""
    return index.get_scores(tokenize(query))


def time_word_tokenize(texts, sample=500, seed=0, limit_minutes=30):
    """Đo underthesea.word_tokenize trên `sample` tài liệu ngẫu nhiên, ước lượng cho cả kho.

    Quy tắc của kế hoạch: dưới 30 phút thì E0 dùng underthesea, ngược lại dùng âm tiết.
    """
    from underthesea import word_tokenize

    picked = random.Random(seed).sample(list(texts), min(sample, len(texts)))
    start = time.perf_counter()
    for text in picked:
        word_tokenize(text)
    seconds = time.perf_counter() - start
    # Ước lượng theo số ký tự, vì độ dài điều luật lệch rất mạnh (có điều dài ~250 nghìn ký tự).
    minutes = seconds / max(sum(map(len, picked)), 1) * sum(map(len, texts)) / 60
    return {'sample_docs': len(picked), 'sample_seconds': seconds, 'docs_per_second': len(picked) / seconds,
            'estimated_minutes': minutes, 'use_underthesea': minutes < limit_minutes}


# ---------- Dense ----------

def embedding_cache_path(cache_dir, dataset, chunking, model_name=DEFAULT_MODEL):
    """Đường dẫn file đệm theo khóa (bộ dữ liệu, cách phân đoạn, mô hình)."""
    return Path(cache_dir) / f'{dataset}__{chunking}__{model_name.replace("/", "_")}.npy'


CHECKPOINT_EVERY = 20_000


def _auto_batch_size(batch_size):
    """Tự chọn batch_size và dtype (bf16 trên A100/L4, fp16 trên T4) theo VRAM GPU; nếu `batch_size > 0` thì giữ nguyên."""
    try:
        import torch
    except ImportError:
        return batch_size or 32, None
    if not torch.cuda.is_available():
        return batch_size or 32, None
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    if batch_size and batch_size > 0:
        return batch_size, dtype
    auto_bs = 512 if vram_gb >= 36 else 256 if vram_gb >= 20 else 64 if vram_gb >= 12 else 32
    return auto_bs, dtype


def _encode_with_oom_backoff(encode_fn, batch, batch_size):
    """Gọi `encode_fn(batch, batch_size)` và tự giảm nửa `batch_size` nếu gặp CUDA OutOfMemoryError."""
    bs = batch_size
    while True:
        try:
            return encode_fn(batch, bs)
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower() or bs <= 8:
                raise
            bs = max(bs // 2, 8)
            try:
                import torch
                torch.cuda.empty_cache()
            except ImportError:
                pass
            print(f'Hết VRAM ở batch_size={bs * 2}, tự giảm xuống batch_size={bs} và chạy lại block...')


def embed(texts, cache_path=None, model_name=DEFAULT_MODEL, batch_size=0, prefix='',
          checkpoint_every=CHECKPOINT_EVERY, encode_fn=None):
    """Nhúng danh sách văn bản thành ma trận float32 chuẩn hóa L2; hỗ trợ checkpoint/resume theo block và fp16/bf16.

    - Nếu `cache_path` đã tồn tại: đọc trực tiếp (tương thích mọi file `.npy` đã lưu từ trước).
    - Nếu `cache_path` được truyền và `len(texts) > checkpoint_every`: lưu từng block `checkpoint_every` văn bản
      vào thư mục `<cache_path>.parts/` để có thể resume khi ngắt phiên hoặc đổi GPU (T4/L4/A100).
    - `batch_size=0` (mặc định): tự chọn theo VRAM GPU (T4=128, L4=256, A100=512).
    """
    cache_path = Path(cache_path) if cache_path is not None else None
    if cache_path is not None and cache_path.exists():
        cached = np.load(cache_path)
        if len(cached) != len(texts):
            raise ValueError(f'File đệm {cache_path} có {len(cached)} dòng nhưng kho có {len(texts)} văn bản; '
                             'hãy xóa file đệm hoặc đổi khóa.')
        return cached

    inputs = [prefix + t for t in texts] if prefix else list(texts)
    bs, dtype = _auto_batch_size(batch_size)

    def _get_encoder():
        if encode_fn is not None:
            return encode_fn
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(model_name)
        model.max_seq_length = 512
        if dtype is not None:
            model.to(dtype=dtype)
            print(f'Đang nhúng trên GPU với dtype={dtype}, batch_size={bs} ({model_name})')
        return lambda batch, cur_bs: model.encode(
            batch, batch_size=cur_bs, normalize_embeddings=True,
            convert_to_numpy=True, show_progress_bar=True).astype(np.float32)

    encoder = None
    start_t = time.perf_counter()

    if cache_path is not None and len(inputs) > checkpoint_every:
        part_dir = cache_path.with_suffix('.parts')
        part_dir.mkdir(parents=True, exist_ok=True)
        shards = []
        for i in range(0, len(inputs), checkpoint_every):
            end = min(i + checkpoint_every, len(inputs))
            shard = part_dir / f'{i:07d}_{end:07d}.npy'
            if shard.exists():
                arr = np.load(shard)
                if len(arr) == end - i:
                    shards.append(arr)
                    continue
            if encoder is None:
                encoder = _get_encoder()
            t0 = time.perf_counter()
            arr = _encode_with_oom_backoff(encoder, inputs[i:end], bs).astype(np.float32)
            tmp = shard.with_suffix('.tmp.npy')
            np.save(tmp, arr)
            tmp.replace(shard)
            shards.append(arr)
            dt = time.perf_counter() - t0
            print(f'[Checkpoint {end}/{len(inputs)}] Lưu {shard.name} ({(end - i) / max(dt, 1e-6):.1f} văn bản/giây)')
        vectors = np.concatenate(shards, axis=0)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        tmp_final = cache_path.with_suffix('.tmp.npy')
        np.save(tmp_final, vectors)
        tmp_final.replace(cache_path)
        for p in part_dir.glob('*.npy'):
            p.unlink(missing_ok=True)
        try:
            part_dir.rmdir()
        except OSError:
            pass
    else:
        encoder = _get_encoder()
        vectors = _encode_with_oom_backoff(encoder, inputs, bs).astype(np.float32)
        if cache_path is not None:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            tmp_final = cache_path.with_suffix('.tmp.npy')
            np.save(tmp_final, vectors)
            tmp_final.replace(cache_path)

    seconds = time.perf_counter() - start_t
    print(f'Đã nhúng {len(texts)} văn bản bằng {model_name} trong {seconds:.1f} giây '
          f'({len(texts) / max(seconds, 1e-6):.1f} tài liệu/giây).')
    return vectors


# ---------- Hybrid và xếp hạng ----------

def minmax(scores):
    """Chuẩn hóa min–max về [0, 1]; vector hằng thì trả về toàn 0."""
    scores = np.asarray(scores, dtype=np.float64)
    span = scores.max() - scores.min()
    return (scores - scores.min()) / span if span > 0 else np.zeros_like(scores)


def hybrid_scores(dense, bm25, alpha=ALPHA):
    """alpha * dense + (1 - alpha) * bm25, mỗi vector điểm được chuẩn hóa min–max trên cả kho."""
    return alpha * minmax(dense) + (1 - alpha) * minmax(bm25)


def top_k(scores, k=TOP_K):
    """Chỉ số của k điểm cao nhất, giảm dần; điểm bằng nhau thì giữ thứ tự trong kho."""
    return np.argsort(-np.asarray(scores), kind='stable')[:k]


# ---------- Bảng kết quả JSONL ----------

def read_rows(path):
    """Đọc các dòng JSONL đã có; bỏ qua dòng cuối bị cắt dở do lượt chạy trước bị ngắt."""
    rows = []
    if Path(path).exists():
        for line in Path(path).read_text(encoding='utf-8').splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def load_jsonl(path):
    """Đọc file JSONL đầu vào (tập đánh giá, kho ViQuAD); dòng hỏng thì báo lỗi chứ không bỏ qua."""
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]


def open_append(path):
    """Mở file để ghi nối tiếp; dòng cuối bị cắt dở thì xuống dòng trước để dòng mới không dính vào."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    dangling = False
    if path.exists() and path.stat().st_size > 0:
        with path.open('rb') as f:
            f.seek(-1, 2)
            dangling = f.read(1) != b'\n'
    f = path.open('a', encoding='utf-8')
    if dangling:
        f.write('\n')
    return f


def run_retrieval(scorers, questions, gold, doc_ids, out_path, k=TOP_K, hit_fn=None):
    """Ghi nối tiếp một dòng cho mỗi cặp (phương pháp, câu hỏi); cặp đã có trong file thì bỏ qua.

    scorers: {tên phương pháp: hàm qid -> vector điểm trên cả kho}. gold[qid] là None với câu không
    trả lời được: vẫn ghi `top` nhưng bốn chỉ số là null. Trả về số dòng mới ghi.
    """
    done = {(r['method'], r['qid']) for r in read_rows(out_path)}
    written = 0
    with open_append(out_path) as f:
        for method, score in scorers.items():
            for qid in questions:
                if (method, qid) in done:
                    continue
                top = [doc_ids[i] for i in top_k(score(qid), k)]
                if gold[qid] is None:
                    scores = dict.fromkeys(METRIC_KEYS)
                elif hit_fn is not None:
                    scores = hit_fn(qid, top)
                else:
                    scores = metrics(zalo_hits(top, gold[qid]), len(gold[qid]))
                row = {'method': method, 'qid': qid, **scores, 'top': top}
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
                f.flush()
                written += 1
    return written


def summarize(out_path, methods):
    """Trả về {phương pháp: {'n': số câu có chỉ số, r5, r20, mrr10, ndcg5}}; dòng chỉ số null không tính."""
    rows = read_rows(out_path)
    table = {}
    for method in methods:
        own = [r for r in rows if r['method'] == method and r['r5'] is not None]
        if own:
            table[method] = {'n': len(own), **mean_metrics(own)}
    return table


def format_table(table):
    header = f'{"Phương pháp":<16}{"n":>6}' + ''.join(f'{name:>11}' for name in
                                                      ('Recall@5', 'Recall@20', 'MRR@10', 'nDCG@5'))
    lines = [f'{method:<16}{row["n"]:>6}' + ''.join(f'{row[k]:>11.4f}' for k in METRIC_KEYS)
             for method, row in table.items()]
    return '\n'.join([header, *lines])


# ---------- CLI ----------

def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument('processed_dir', help='thư mục zalo_legal đã xử lý, hoặc corpus.jsonl của ViQuAD')
    parser.add_argument('out_dir')
    parser.add_argument('--methods', default='bm25,dense,hybrid')
    parser.add_argument('--alpha', type=float, default=ALPHA)
    parser.add_argument('--model', default=DEFAULT_MODEL)
    parser.add_argument('--batch-size', type=int, default=0,
                        help='0 = tự chọn theo VRAM GPU (T4=128, L4=256, A100=512)')
    parser.add_argument('--tokenizer', choices=['syllable', 'word'], default='syllable',
                        help='cách tách cho BM25: âm tiết, hoặc từ bằng underthesea (đệm trong <out_dir>/tokens)')
    parser.add_argument('--questions', help='test.jsonl hoặc dev.jsonl của tập đánh giá đã đóng băng')
    parser.add_argument('--removed', help='removed_doc_ids.json: các tài liệu loại khỏi kho trước khi truy xuất')
    parser.add_argument('--chunking', choices=['c1', 'c3', 'c4', 'c5', 'c5_nohead'],
                        help='chạy ở mức chunk (mặc định: mức tài liệu/đoạn)')
    parser.add_argument('--run', help='nhãn lượt chạy, kết quả ghi vào retrieval_<run>.jsonl (mặc định zalo)')
    parser.add_argument('--time-tokenizer', action='store_true',
                        help='đo tốc độ underthesea.word_tokenize trên 500 điều luật rồi thoát')
    args = parser.parse_args(argv)

    from ragvi.data import expand_zalo_doc_ids, load_zalo_corpus, load_zalo_queries

    if args.processed_dir.endswith('.jsonl'):
        # ponytail: đoạn ViQuAD chỉ lấy `text`, không ghép title (nhiều đoạn chung một title; vị trí đáp án tính trên text).
        corpus = {r['doc_id']: r['text'] for r in load_jsonl(args.processed_dir)}
        titles = None
        dataset, base_unit = 'viquad', 'passage'
        if args.chunking in ('c5', 'c5_nohead'):
            raise SystemExit('ViQuAD chỉ dùng c1, c3, c4.')
    else:
        corpus = load_zalo_corpus(args.processed_dir)
        titles = None
        if args.chunking in ('c5', 'c5_nohead') and (Path(args.processed_dir) / 'corpus').is_dir():
            from datasets import load_from_disk
            ds_corpus = load_from_disk(str(Path(args.processed_dir) / 'corpus')).select_columns(['id', 'title'])
            titles = {row['id']: row['title'] for row in ds_corpus}
        dataset, base_unit = 'zalo_legal', 'article'
    unit = args.chunking or base_unit
    doc_ids, texts = list(corpus), list(corpus.values())
    if args.time_tokenizer:
        print(json.dumps(time_word_tokenize(texts), ensure_ascii=False, indent=2))
        return

    methods = args.methods.split(',')
    unknown = set(methods) - {'bm25', 'dense', 'hybrid'}
    if unknown:
        raise SystemExit(f'Phương pháp không hợp lệ: {sorted(unknown)}')

    out = Path(args.out_dir)
    raw_removed = set(json.loads(Path(args.removed).read_text(encoding='utf-8'))) if args.removed else set()
    if raw_removed - set(corpus):
        raise SystemExit(f'{len(raw_removed - set(corpus))} doc_id trong {args.removed} không có trong kho.')
    removed = expand_zalo_doc_ids(raw_removed, corpus) if raw_removed else set()

    if args.chunking:
        chunks = cached_chunks(corpus, args.chunking, out / 'chunks' / f'{dataset}__{args.chunking}.jsonl',
                               titles=titles)
        unit_ids, unit_texts = [c[0] for c in chunks], [c[4] for c in chunks]
        parent_of = {c[0]: c[1] for c in chunks}
        span_of = {c[0]: (c[1], c[2], c[3]) for c in chunks}
        keep = [i for i, c in enumerate(chunks) if c[1] not in removed]
    else:
        unit_ids, unit_texts = doc_ids, texts
        parent_of = {d: d for d in doc_ids}
        span_of = {d: (d, 0, len(corpus[d])) for d in doc_ids}
        keep = [i for i, d in enumerate(doc_ids) if d not in removed]
    kept_ids, kept_texts = [unit_ids[i] for i in keep], [unit_texts[i] for i in keep]
    kept_docs = {parent_of[u] for u in kept_ids}

    answers_of = {}
    if args.questions:
        if args.run in (None, 'zalo'):
            raise SystemExit('Chạy trên tập đánh giá phải có --run khác "zalo" để không lẫn với retrieval_zalo.jsonl.')
        rows = load_jsonl(args.questions)
        questions = {q['qid']: q['question'] for q in rows}
        gold = {q['qid']: (set(q['gold_doc_ids']) if 'gold_doc_ids' in q else {q['gold_doc_id']})
                if q['answerable'] else None for q in rows}
        answers_of = {q['qid']: q.get('answers') or [] for q in rows}
        lost = set().union(*filter(None, gold.values())) - kept_docs
        if lost:
            raise SystemExit(f'{len(lost)} tài liệu vàng của câu trả lời được không có trong kho, ví dụ {sorted(lost)[:3]}.')
    elif dataset == 'viquad' or removed:
        raise SystemExit('Kho ViQuAD hoặc --removed cần đi kèm --questions.')
    else:
        questions, gold = load_zalo_queries(args.processed_dir)
    run = args.run or 'zalo'
    out_path = out / f'retrieval_{run}.jsonl'

    # Tham số khác mặc định thì đổi nhãn, để dòng cũ trong JSONL không bị coi là đã xong.
    chunk_tag = f'[{args.chunking}]' if args.chunking else ''
    model_tag = '' if args.model == DEFAULT_MODEL else f'[{args.model}]'
    word_tag = '[word]' if args.tokenizer == 'word' else ''
    labels = {'bm25': 'bm25' + chunk_tag + word_tag, 'dense': 'dense' + chunk_tag + model_tag,
              'hybrid': 'hybrid' + chunk_tag + word_tag + model_tag +
              ('' if args.alpha == ALPHA else f'[alpha={args.alpha:g}]')}
    done = {(r['method'], r['qid']) for r in read_rows(out_path)}
    pending = [m for m in methods if any((labels[m], q) not in done for q in questions)]

    tokenize = tokenize_words if args.tokenizer == 'word' else tokenize_syllables
    if {'bm25', 'hybrid'} & set(pending):
        if args.tokenizer == 'word':
            tok_path = out / 'tokens' / f'{dataset}__{unit}__underthesea.jsonl'
            base_tok_path = out / 'tokens' / f'{dataset}__{base_unit}__underthesea.jsonl'
            if args.chunking == 'c4' and not tok_path.exists() and base_tok_path.exists():
                tokens = cached_tokens([parent_of[u] for u in kept_ids], kept_texts, base_tok_path)
            else:
                tokens = cached_tokens(kept_ids, kept_texts, tok_path)
            index = build_bm25(tokens, tokenize=lambda t: t)
        else:
            index = build_bm25(kept_texts)
    if {'dense', 'hybrid'} & set(pending):
        cache = out / 'embeddings'
        q_pref, p_pref = model_prefixes(args.model)
        emb_path = embedding_cache_path(cache, dataset, unit, args.model)
        base_emb_path = embedding_cache_path(cache, dataset, base_unit, args.model)
        if args.chunking == 'c4' and not emb_path.exists() and base_emb_path.exists():
            emb_path = base_emb_path
        doc_vectors = embed(unit_texts, emb_path, args.model, args.batch_size, prefix=p_pref)
        if removed:
            doc_vectors = doc_vectors[keep]
        query_vectors = embed(list(questions.values()),
                              embedding_cache_path(cache, dataset, 'queries' if run == 'zalo' else f'queries_{run}',
                                                   args.model),
                              args.model, args.batch_size, prefix=q_pref)
        row_of = {qid: i for i, qid in enumerate(questions)}

    def bm25(qid):
        return bm25_scores(index, questions[qid], tokenize)

    def dense(qid):
        return doc_vectors @ query_vectors[row_of[qid]]

    def hybrid(qid):
        return hybrid_scores(dense(qid), bm25(qid), args.alpha)

    def hit_fn(qid, top):
        if dataset == 'viquad' and args.chunking:
            return metrics(viquad_hits([span_of[c] for c in top], next(iter(gold[qid])), answers_of[qid]), 1)
        return metrics(zalo_hits([parent_of[c] for c in top], gold[qid]), len(gold[qid]))

    scorers = {'bm25': bm25, 'dense': dense, 'hybrid': hybrid}
    written = run_retrieval({labels[m]: scorers[m] for m in pending}, questions, gold, kept_ids, out_path,
                            hit_fn=hit_fn if args.chunking else None)
    print(f'{len(kept_ids)} tài liệu, {len(questions)} truy vấn; ghi thêm {written} dòng vào {out_path}')
    print(format_table(summarize(out_path, [labels[m] for m in methods])))


if __name__ == '__main__':
    main()


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/review.py`

Chép mã nguồn mới nhất (có checkpoint/resume embedding + fp16/bf16) vào `/content/src/ragvi/`.

In [ ]:
%%writefile /content/src/ragvi/review.py
"""Dựng phiếu đọc tay câu "không trả lời được" (quyết định D6) và đếm phiếu đã chấm.

Chạy từ thư mục gốc của kho mã:
    PYTHONPATH=src python -m ragvi.review <retrieval.jsonl> <test.jsonl> <kho> <out.md> [--method hybrid] [--n 30] [--seed 0]
    PYTHONPATH=src python -m ragvi.review --tally <out.md đã chấm>
<kho> là thư mục <processed>/zalo_legal hoặc file corpus.jsonl của ViQuAD.
"""
import argparse
import json
import random
import re
import sys
from pathlib import Path

THRESHOLD = 0.20
MAX_CHARS = 1200
BOX_NO, BOX_YES = 'không trả lời được', 'VẪN trả lời được'

HEADER = f"""# Phiếu đọc tay câu "không trả lời được" (D6)

Với mỗi câu, đọc câu hỏi và 5 đoạn đầu của truy xuất rồi tự hỏi: "Chỉ dựa vào 5 đoạn dưới đây, có trả lời đúng và đầy đủ được câu hỏi không?"
Nếu không, đánh `[x]` vào ô "{BOX_NO}"; nếu có, đánh `[x]` vào ô "{BOX_YES}" và ghi số thứ hạng của đoạn chứa đáp án.
Chỉ đánh đúng một ô mỗi câu; câu phân vân thì ghi lý do vào Ghi chú.
Đoạn có nhãn **[ĐOẠN GỐC]** là đoạn vàng của câu hỏi (ViQuAD: đoạn đối kháng mà câu hỏi được viết dựa vào; Zalo: điều luật lẽ ra đã bị loại khỏi kho).
Nếu tỷ lệ "{BOX_YES}" trên {THRESHOLD:.0%}, phải thay các câu đó trước khi đóng băng tập test.
Đếm kết quả: `PYTHONPATH=src python -m ragvi.review --tally <file này>`.
"""


def pick(test_rows, results, n=30, seed=0):
    """Chọn ngẫu nhiên theo seed n câu answerable=False có trong results ({qid: top}).

    Trả về danh sách câu hỏi theo thứ tự đọc.
    """
    pool = sorted((q for q in test_rows if not q['answerable']), key=lambda q: q['qid'])
    if not pool:
        raise ValueError('Tập test không có câu "không trả lời được" nào.')
    with_result = [q for q in pool if q['qid'] in results]
    if len(with_result) < n:
        raise ValueError(f'Chỉ {len(with_result)}/{len(pool)} câu không trả lời được có kết quả truy xuất, '
                         f'cần {n}. Kiểm tra --method hoặc chạy lại truy xuất cho đủ.')
    return random.Random(seed).sample(with_result, n)


def render(picked, results, corpus, k=5, max_chars=MAX_CHARS):
    """Trả về nội dung Markdown của phiếu. corpus: {doc_id: (title, text)}."""
    out = [HEADER]
    for i, q in enumerate(picked, 1):
        gold = set(q.get('gold_doc_ids') or [q.get('gold_doc_id')])
        out.append(f'\n---\n\n## Câu {i} — `{q["qid"]}`\n\n**Câu hỏi:** {q["question"]}\n')
        for rank, doc_id in enumerate(results[q['qid']][:k], 1):
            title, text = corpus.get(doc_id, ('(không có trong kho)', ''))
            if len(text) > max_chars:
                text = text[:max_chars].rstrip() + ' … (đã cắt)'
            mark = ' **[ĐOẠN GỐC]**' if doc_id in gold else ''
            body = '\n'.join('> ' + line for line in text.splitlines()) or '>'
            out.append(f'\n**Đoạn {rank}** — `{doc_id}`{mark} — {title}\n\n{body}\n')
        out.append(f'\nKết luận: [ ] {BOX_NO}   [ ] {BOX_YES} (đoạn số: __)   Ghi chú:\n')
    out.append('\n---\n\n## Tổng kết\n\n| Số câu đã đọc | Số câu vẫn trả lời được | Tỷ lệ |\n|---|---|---|\n|  |  |  |\n')
    return ''.join(out)


def tally(markdown):
    """Đếm phiếu đã chấm. Trả về dict: total, yes, no, unmarked [qid], both [qid], rate, over."""
    res = {'total': 0, 'yes': [], 'no': [], 'unmarked': [], 'both': []}
    for block in re.split(r'^## Câu \d+ — ', markdown, flags=re.M)[1:]:
        qid = block.split('`')[1]
        line = next((l for l in block.splitlines() if l.startswith('Kết luận:')), '')
        no = re.search(r'\[[xX]\]\s*' + BOX_NO, line) is not None
        yes = re.search(r'\[[xX]\]\s*' + BOX_YES, line) is not None
        res['total'] += 1
        res['both' if no and yes else 'yes' if yes else 'no' if no else 'unmarked'].append(qid)
    graded = len(res['yes']) + len(res['no'])
    res['rate'] = len(res['yes']) / graded if graded else None
    res['over'] = res['rate'] is not None and res['rate'] > THRESHOLD
    return res


def read_jsonl(path):
    """Đọc JSONL, bỏ qua dòng không parse được (dòng cuối có thể bị cắt dở)."""
    rows = []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    return rows


def load_corpus(path):
    """Trả về {doc_id: (title, text)} từ corpus.jsonl ViQuAD hoặc thư mục Zalo đã xử lý."""
    if str(path).endswith('.jsonl'):
        return {r['doc_id']: (r['title'], r['text']) for r in read_jsonl(path)}
    from ragvi.data import load_zalo_corpus
    return {d: tuple(v.split('\n', 1)) if '\n' in v else (v, '') for d, v in load_zalo_corpus(path).items()}


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument('paths', nargs='*', help='<retrieval.jsonl> <test.jsonl> <kho> <out.md>')
    p.add_argument('--method', default='hybrid')
    p.add_argument('--n', type=int, default=30)
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--tally', metavar='PHIEU_DA_CHAM')
    a = p.parse_args(argv)

    if a.tally:
        r = tally(Path(a.tally).read_text(encoding='utf-8'))
        graded = len(r['yes']) + len(r['no'])
        print(f'Số câu trong phiếu: {r["total"]}; đã chấm: {graded}; vẫn trả lời được: {len(r["yes"])}')
        if r['unmarked']:
            print('Chưa chấm:', ', '.join(r['unmarked']))
        if r['both']:
            print('Chấm cả hai ô (cần sửa):', ', '.join(r['both']))
        if r['rate'] is None:
            print('Chưa có câu nào chấm hợp lệ.')
        else:
            verdict = 'VƯỢT ngưỡng, phải thay câu' if r['over'] else 'không vượt ngưỡng'
            print(f'Tỷ lệ vẫn trả lời được: {len(r["yes"])}/{graded} = {r["rate"]:.1%} ({verdict} {THRESHOLD:.0%})')
            if r['unmarked'] or r['both']:
                print('Lưu ý: tỷ lệ chỉ tính trên các câu đã chấm hợp lệ; hoàn tất phiếu trước khi ghi vào D6.')
        return
    if len(a.paths) != 4:
        p.error('cần đủ 4 đường dẫn: <retrieval.jsonl> <test.jsonl> <kho> <out.md>, hoặc dùng --tally')
    retrieval, test, corpus_path, out = a.paths
    if Path(out).exists():
        sys.exit(f'{out} đã tồn tại; không ghi đè phiếu có thể đã chấm. Chọn tên khác.')
    results = {r['qid']: r['top'] for r in read_jsonl(retrieval) if r.get('method') == a.method and 'top' in r}
    if not results:
        sys.exit(f'Không có dòng kết quả nào với method={a.method!r} trong {retrieval}.')
    try:
        picked = pick(read_jsonl(test), results, a.n, a.seed)
    except ValueError as e:
        sys.exit(str(e))
    Path(out).write_text(render(picked, results, load_corpus(corpus_path)), encoding='utf-8')
    print(f'Đã ghi {out}: {len(picked)} câu, method={a.method}, seed={a.seed}')


if __name__ == '__main__':
    main()


### [TÁI SỬ DỤNG] Hàm chạy CLI và ghi log

Đặt `PYTHONPATH` và định nghĩa `run()` để gọi CLI và ghi log vào `O2/logs/`.

In [ ]:
import os
import subprocess
import sys

os.environ['PYTHONPATH'] = '/content/src'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
if '/content/src' not in sys.path:
    sys.path.insert(0, '/content/src')
LOG = O2 / 'logs'
LOG.mkdir(parents=True, exist_ok=True)


def run(name, module, *args):
    cmd = [sys.executable, '-m', module, *map(str, args)]
    print('$', ' '.join(cmd))
    with open(LOG / f'{name}.log', 'w', encoding='utf-8') as log, subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
    if proc.returncode:
        raise RuntimeError(f'Lệnh {name} lỗi (mã thoát {proc.returncode}); xem {LOG / name}.log')

### [CHẠY MỘT LẦN / KHI CẬP NHẬT] Ngày 11–12 (ưu tiên chạy trước): Judge trên dev (`Qwen/Qwen3-8B`) và xuất 2 phiếu gán nhãn tay 100 câu

Chạy trước phần Judge (~15–20 phút) để có ngay 2 phiếu gán nhãn tay ẩn cấu hình (`human_dev_viquad.md`, `human_dev_zalo.md`) và kiểm tra JSON hợp lệ + chỉ số nhầm điều luật trên dev trước khi treo máy nhúng C5.

In [ ]:
for ds, corpus_unit, ds_key in [('viquad', EVAL / 'viquad' / 'corpus.jsonl', 'viquad'),
                                ('zalo', Z, 'zalo_legal')]:
    ans_path = GEN / f'dev_{ds}.jsonl'
    q_dev = EVAL / ds / 'dev.jsonl'
    c3_chunks = R / 'chunks' / f'{ds_key}__c3.jsonl'
    sheet = SHEETS2 / f'human_dev_{ds}.md'
    if not sheet.exists():
        run(f'sheet_{ds}', 'ragvi.judge', ans_path, q_dev, corpus_unit, sheet,
            '--chunks', c3_chunks, '--human-sheet', '--n', '50', '--seed', '0')
    run(f'judge_{ds}_dev20', 'ragvi.judge', ans_path, q_dev, corpus_unit,
        JUDGE / f'judge_dev_{ds}.jsonl', '--chunks', c3_chunks, '--model', JUDGE_MODEL, '--limit', '20')
    run(f'judge_{ds}_dev_all', 'ragvi.judge', ans_path, q_dev, corpus_unit,
        JUDGE / f'judge_dev_{ds}.jsonl', '--chunks', c3_chunks, '--model', JUDGE_MODEL)

### [CHẠY MỘT LẦN / KHI CẬP NHẬT] Ngày 10 (phần còn lại): Nhúng nốt `c5` và `c5_nohead` trên Zalo (có checkpoint mỗi 20.000 chunk + bf16/fp16)

6/8 lượt `BAAI/bge-m3` đã xong ở notebook 04. Cell này nhúng nốt `c5` và `c5_nohead` với `bf16`/`fp16` và `batch_size` tự động theo GPU (T4=128, L4=256, A100=512). Mỗi 20.000 chunk lưu một shard vào `.parts/` trên Drive: nếu ngắt phiên hoặc đổi GPU chỉ cần chạy lại cell này sẽ tự nối tiếp.

In [ ]:
for c in ('c5', 'c5_nohead'):
    run(f'ret_zalo_dense_{c}', 'ragvi.retrieval', Z, R,
        '--run', 'zalo_phase_a', '--chunking', c, '--methods', 'dense',
        '--questions', EVAL / 'zalo' / 'test.jsonl',
        '--removed', EVAL / 'zalo' / 'removed_doc_ids.json')

### [TÁI SỬ DỤNG] Tổng hợp kết quả Tuần 2 và in 2 phiếu gán nhãn tay ra output

In toàn bộ bảng truy xuất Giai đoạn A (`bge-m3`), tổng hợp Judge trên dev và xuất nội dung 2 phiếu gán nhãn tay.

In [ ]:
import json
from ragvi.retrieval import format_table, read_rows, summarize

for title, name in [('ViQuAD Phase A (dense bge-m3)', 'retrieval_viquad_phase_a.jsonl'),
                    ('Zalo Phase A (dense bge-m3)', 'retrieval_zalo_phase_a.jsonl')]:
    p = R / name
    if p.exists():
        methods = sorted({r['method'] for r in read_rows(p)})
        print(f'\n== {title} ==\n' + format_table(summarize(p, methods)))

print('\n=== NGÀY 11: TỔNG HỢP JUDGE TRÊN DEV ===')
for ds in ('viquad', 'zalo'):
    p = JUDGE / f'judge_dev_{ds}.jsonl'
    if p.exists():
        rows = read_rows(p)
        valid = [r for r in rows if r.get('error') is None]
        non_ref = [r for r in valid if not r.get('refused')]
        wrong_art = [r for r in non_ref if 'wrong_article' in r]
        print(ds, json.dumps({
            'total': len(rows), 'valid_json': len(valid), 'errors': len(rows) - len(valid),
            'h_gold_mean': sum(r['h_gold'] for r in non_ref) / len(non_ref) if non_ref else None,
            'correctness_mean': sum(r['correctness'] for r in valid) / len(valid) if valid else None,
            'wrong_article_rate': sum(bool(r['wrong_article']) for r in wrong_art) / len(wrong_art) if wrong_art else None,
        }, ensure_ascii=False))

for name in ('human_dev_viquad.md', 'human_dev_zalo.md'):
    path = SHEETS2 / name
    print(f'<<<SHEET {name}>>>')
    print(path.read_text(encoding='utf-8') if path.exists() else '(chưa có)')
    print(f'<<<END {name}>>>')